# 会说话的主播：一张角色图 + 一段文稿 → 口播视频

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dawang-sakura/colab-talking-avatar/blob/main/talking_avatar.ipynb)

```
文稿 ──VoxCPM2 配音──▶ 语音 wav ──ComfyUI + InfiniteTalk 对嘴──▶ 无声视频 ──合成＋烧字幕──▶ talking_avatar.mp4
```

全程在 Google Colab 的 **L4 GPU** 上运行，你的电脑不用安装任何东西。

**使用步骤**
1. 菜单「代码执行程序 → 更改运行时类型」（英文界面：Runtime → Change runtime type）选 **L4 GPU** → 保存。
2. 改下面三格（都不改也能直接跑示例）：
   - **① 文稿**：你的台词，直接改程序码里三个引号之间的句子。
   - **①-2 设置**：声音、角色图、画面比例、去背、版面、字幕，都在表单里选（下拉选单、滑杆、勾选框、文字框）。
   - **①-3 上传文件**（选填）：要用自己的角色图、录影、BGM、字体、参考录音时勾选，运行到这一格会出现「选择文件」按钮。
3. 「代码执行程序 → 全部运行」（英文界面：Runtime → Run all）。弹出「此笔记本不是由 Google 创作」时点「仍然运行」。每格做完会显示绿色「✅」框；很久的步骤（③-1 配音、④-2 下载模型、④-5 对嘴）开头会显示黄色「⏳ 进行中」框。
4. 跑完浏览器会自动下载一个 `talking_avatar_outputs.zip`（一次流程只下载这一个文件），解压后是 `talking_avatar.mp4`（默认带字幕），另附字幕文件 `talking_avatar.srt` 和不带字幕的 `talking_avatar_nosub.mp4`；①-2 开了去背（`BG_REMOVE`）时再加一份透明背景的 `talking_avatar_alpha.webm`（选「绿幕」时还有对嘴用的绿幕图 `input_greenscreen.png`）。字幕和去背都关时直接下载 `talking_avatar.mp4`。①-2 的 `LAYOUT` 选「上主播下录影」时，成品是 1080×1920 的上下合成版，zip 里另有只有主播的 `talking_avatar_host.mp4`（作者在 Colab L4 上实测过一次）。
5. **用完务必「代码执行程序 → 断开连接并删除运行时」**（英文界面：Runtime → Disconnect and delete runtime），闲置也会扣计算单元。

**跑完后只改一部分：改哪格、跑哪格**（要在删除运行时之前做，不用重跑配音和对嘴）

| 想改什么 | 改哪里 | 然后依次运行 |
|---|---|---|
| 字幕样式、字幕开关 | ①-2「字幕」（自定义字体也可以用 ①-3 上传） | ①-2（用了 ①-3 再运行 ①-3）→ ⑥ |
| 版面微调（`REC_CROP_TOP`、`REC_CROP_BOTTOM`、`REC_CENTER_X`、`HOST_CROP_TOP`、`BG_BLUR`、`BG_DIM`、`REC_AUDIO`） | ①-2「版面」 | ①-2 → ⑤ |
| 换 BGM | ①-2 的 `BGM_PATH`，或 ①-3 上传 | ①-2（用了 ①-3 再运行 ①-3）→ ②-2 → ⑤ |

文稿、声音、角色图、画面比例、去背、录影改了都要「全部运行」（会重新配音和对嘴）。中途「重启会话」过：①-3 上传过的文件会自动沿用，不用重新上传。

> **使用规范**：只能使用你本人、已取得当事人同意、或 AI 生成的虚构角色的脸和声音；禁止冒充他人。Colab 明文禁止 creating deepfakes（[Colab FAQ](https://research.google.com/colaboratory/faq.html)）。

没用过 Colab？请看仓库里的《新手操作手册》：`docs/新手操作手册.md`。

## ① 文稿
**在这一格的程序码里改台词**：换掉三个引号 `"""` 之间的句子（引号要留着）。其他设置都在下一格 **①-2 设置**。改了文稿要「全部运行」（会重新配音和对嘴）。
- 一行一句；每句最多 80 字，超过会自动切开（VoxCPM2 单次太长容易出现啸叫或音色漂移）。
- 句首可以用括号写语气，例如 `(轻松地)大家好。` 开启「极致克隆」（①-2 的「进阶：声音克隆」）时不能加语气括号。
- 每行开头可以写开始秒数（选填），例如 `14.0｜(兴奋)来了来了！`（全角 `｜` 或半角 `|` 都可以）：这句从第 14.0 秒开始念；没写秒数的行接在上一句后面（隔 `PAUSE_SEC` 秒）；指定的时间上一句还没念完时会往后顺延并提醒。主要给 ①-2「版面」选「上主播下录影」时对准录影画面用。

In [ ]:
# 在三个引号之间改台词，引号要留着
SCRIPT = """
大家好，我是虚构的 AI 主播。
一张图加一段文稿，就能让我开口说话。
全程在 Colab 上运行，不用装任何东西。
"""

In [ ]:
# @title ①-2 设置 { display-mode: "form" }
# @markdown 声音、角色图、画面比例、去背、版面、字幕都在这里选，**不改也能直接跑示例**。改完要运行本格（或「全部运行」）才生效，本格几秒就好。跑完后只改一部分时「改哪格、跑哪格」见第一格的表。
# @markdown 用自己的文件（角色图、录影、BGM、字体、参考录音）：在下一格 **①-3 上传文件** 勾选上传；或先在左侧「文件」面板上传（会放在 `/content/`），再在这里填路径。**同一项两边都有时**：①-3 上传的文件优先，重跑本格也不会丢；之后在这里把那个路径改成别的值＝改用你填的路径（上传记录作废）。
# @markdown ### 声音
# @markdown 声音来源的优先顺序：1. 开启了下方「进阶：声音克隆」→ 用你的录音；2. 否则 `VOICE` 选了声音包里的声音 → 用仓库 `voices/` 里的参考音（试听见 README 的「声音包」）；3. `VOICE` 选「自定义（用 VOICE_DESC）」→ 照 `VOICE_DESC` 的文字描述做音色设计。
VOICE = "新闻女主播"  #@param ["新闻女主播", "温柔知性", "元气少女", "甜美邻家", "成熟御姐", "热情带货", "沉稳旁白", "阳光青年", "亲切大叔", "激情解说", "自定义（用 VOICE_DESC）"]
# @markdown `VOICE_DESC`：音色设计用的文字描述（`VOICE` 选「自定义」时才用）
VOICE_DESC = "年轻女性，声音清亮自然，语速中等，语气亲切平稳，像新闻主播在播报"  #@param {type:"string"}
# @markdown `PAUSE_SEC`：句与句之间的停顿（秒）
PAUSE_SEC = 0.4  #@param {type:"number"}
# @markdown `SEED`：随机种子，音色设计、逐句配音、对嘴生成都用它；对声音不满意可以换一个数字重跑。用声音包时，音色由参考音固定，`SEED` 只影响每句的细微差异和对嘴画面
SEED = 42  #@param {type:"integer"}
# @markdown ### 角色图
# @markdown `IMAGE_PATH`：默认用仓库自带的示例图（AI 生成的虚构人物）。换成自己的图：在 ①-3 勾「角色图」上传；或上传到左侧「文件」面板（会放在 `/content/`），再改成 `/content/你的图.png`
IMAGE_PATH = "examples/host.png"  #@param {type:"string"}
# @markdown `PROMPT`：对嘴画面的英文描述，可按角色外观补充（例如发型、服装）
PROMPT = "A person is talking to the camera, facing the camera, upper body, natural head movement, clear mouth movement."  #@param {type:"string"}
# @markdown ### 画面比例
# @markdown `ASPECT`：成品的画面比例；像素总量都接近 480×832，长宽都是 16 的倍数。作者实测过 9:16 和 1:1，16:9、4:5、3:4 未实测
ASPECT = "9:16 竖屏短视频（480×832）"  #@param ["9:16 竖屏短视频（480×832）", "16:9 横屏视频（832×480）", "1:1 方形帖子（624×624）", "4:5 Instagram 竖版帖子（576×720）", "3:4 小红书竖版（576×768）"]
# @markdown `CROP_POS`：角色图比例和画面不同时先裁切再缩放，裁切位置：
# @markdown - 偏上＝上下要裁时保留上方（头部），左右要裁时左右各裁一半
# @markdown - 置中＝上下、左右都保留中间
# @markdown - 偏左／偏右＝左右要裁时保留左边／右边；上下要裁时和「偏上」一样保留上方
CROP_POS = "偏上（保留头部）"  #@param ["偏上（保留头部）", "置中", "偏左", "偏右"]
# @markdown ### 去背（默认关闭）
# @markdown `BG_REMOVE`：开启后另外输出一份透明背景的 `talking_avatar_alpha.webm`（VP9＋透明通道，不烧字幕），给之后放进别的模板合成用
# @markdown - 关＝不去背
# @markdown - 绿幕（推荐）＝对嘴前先把角色图的背景换成纯绿色，对嘴完再用 ffmpeg 把绿色扣掉；只多处理一张图。注意：这时 `talking_avatar.mp4` 的背景是绿色的；头发边缘可能带绿边，遇到时改用「逐帧」
# @markdown - 逐帧＝角色图照原样对嘴，完成后对成品的每一帧去背；比绿幕慢，前后帧的边缘可能轻微闪动
# @markdown
# @markdown 去背工具：BiRefNet（MIT 授权），运行时从 Hugging Face 官方仓库下载
BG_REMOVE = "关"  #@param ["关", "绿幕（推荐）", "逐帧"]
# @markdown ### 版面（默认「只有主播」）
# @markdown `LAYOUT`：
# @markdown - 只有主播＝和以前一样，只出主播的口播视频
# @markdown - 上主播下录影＝成品固定 1080×1920（9:16），切成上下两格、直接相接（没有渐层、没有外框）：上格 1080×820 是主播（头顶贴近上缘、露到胸口），下格 1080×1100 是你上传的录影，字幕放在两格交界。录影先按 `REC_CROP_TOP`／`REC_CROP_BOTTOM` 裁掉不属于游戏的部分（例如浏览器网址列），剩下的高度整段放进下格、上下不再裁；左右按下格的比例以 `REC_CENTER_X` 为中心取。录影比下格还窄时（例如已经裁好的直式录影）照高度放满，两侧用同一支录影放大、模糊、压暗补满。成片长度跟着录影走；台词可以在行首写开始秒数对准录影画面（见 ① 文稿）。
# @markdown
# @markdown 这个版面用主播的原背景，`BG_REMOVE` 要选「关」；画面比例建议用 9:16。切两格的合成只在作者电脑上验证过，还没在 Colab 上实跑
LAYOUT = "只有主播"  #@param ["只有主播", "上主播下录影"]
# @markdown **以下选「上主播下录影」时才用：**
# @markdown `REC_PATH`：录影，mp4、mov、webm、mkv 等常见格式都可以。在 ①-3 勾「录影」上传；或上传到左侧「文件」面板（会放在 `/content/`）后填路径（录影很大时建议用文件面板）
REC_PATH = "/content/my_recording.mp4"  #@param {type:"string"}
# @markdown `REC_CROP_TOP`／`REC_CROP_BOTTOM`：录影上方／下方裁掉的比例（%），只用来去掉不属于游戏的部分；0＝不裁
# @markdown - 例：电脑浏览器录屏上方的网址列（1920×1040 的录屏，网址列 84 像素，约 8%）、手机录屏上方的状态栏
# @markdown - 游戏下方的操作列（调速度、调额度等按钮）不要裁：解说操作时要看得到
# @markdown - 下格上缘还露出一条网址列 → `REC_CROP_TOP` 调大；游戏上方被切掉 → 调小（每 1% 约是录影高度的百分之一）
REC_CROP_TOP = 0  #@param {type:"slider", min:0, max:45, step:1}
REC_CROP_BOTTOM = 0  #@param {type:"slider", min:0, max:45, step:1}
# @markdown `REC_CENTER_X`：游戏在录影里左右的中心位置（%）：50＝正中间（网页游戏的录屏一般在正中间）；游戏偏左就调小，偏右就调大。左右取多宽是自动算的：裁完上下后剩下的高度 × 1080 ÷ 1100；取到录影边缘外时会自动往内挪并提醒。录影比下格窄时不用这个值
REC_CENTER_X = 50  #@param {type:"slider", min:0, max:100, step:1}
# @markdown `HOST_CROP_TOP`：主播片上方裁掉的比例（%）：主播片等比放大到 1080 宽后，从这个高度往下取 820 像素放进上格
# @markdown - 默认 9：一般「人物头在上方」的角色图（④-3 按「偏上」裁好的 9:16 图），头顶会贴近上缘、身体露到胸口
# @markdown - 头顶上方空太多 → 调大（例如 12～14）；头顶被切掉 → 调小（0＝从主播片最上面开始取）
HOST_CROP_TOP = 9  #@param {type:"slider", min:0, max:40, step:1}
# @markdown `REC_AUDIO`：保留录影原音（录影本来没有声音就没有）；主播讲话时会自动压低
REC_AUDIO = True  #@param {type:"boolean"}
# @markdown `BGM_PATH`：背景音乐（选填）：在 ①-3 勾「BGM」上传，或上传后填路径，例如 `/content/bgm.mp3`；留空＝不加。比成片短时循环播放，主播讲话时自动压低。请自行确认音乐的使用授权
BGM_PATH = ""  #@param {type:"string"}
# @markdown `BG_BLUR`／`BG_DIM`（进阶，一般不用改）：录影比下格窄、两侧要补边时，补边（同一支录影放大）的模糊程度（不能小于 0）和亮度倍率（0～1，越小越暗）
BG_BLUR = 7  #@param {type:"number"}
BG_DIM = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
# @markdown 跑完后只想调上面这些版面设置：改好 → 运行本格 → 只运行 ⑤，不用重跑配音和对嘴（换了 BGM 时，本格之后先运行 ②-2 再运行 ⑤；见 ⑤ 的说明；只在作者电脑上模拟验证过）
# @markdown ### 字幕（默认开启）
# @markdown 字幕文字直接用 ① 的文稿原文，时间按每句配音的实际长度推算（不做语音识别，所以不会有错字）。一句太长、一行放不下时会自动切成几段轮流显示，每段时间按字数比例分配。成品会把字幕烧进画面，另外再给一份 `talking_avatar.srt`。
# @markdown
# @markdown **跑完想换字幕样式**：改这一段 → 运行本格 → 运行「⑥ 只重做字幕」格，不用重跑配音和对嘴
SUBTITLE = True  #@param {type:"boolean"}
# @markdown `SUB_NO_COMMA_PERIOD`：字幕去掉逗号和句号（默认开）：句尾的 ，。,. 删掉，句中的换成一个空格；！？等其他标点照留，数字里的小数点和千分位逗号（2.5、1,000）也照留。只影响字幕（烧进画面的字幕和 `talking_avatar.srt`），配音照文稿念；关掉＝字幕照文稿原样显示标点
SUB_NO_COMMA_PERIOD = True  #@param {type:"boolean"}
# @markdown `SUB_FONT`：都是可免费使用的开源字体，运行时从官方发布处下载（各款授权见 README「字幕」；霞鹜新晰黑是 IPA 授权，其余是 SIL OFL）
SUB_FONT = "思源黑体（粗）"  #@param ["思源黑体（粗）", "站酷快乐体", "未来荧黑 Heavy", "得意黑", "寒蝉圆黑体", "霞鹜新晰黑", "霞鹜文楷", "自定义（上传字体文件）"]
# @markdown `SUB_FONT_FILE`：`SUB_FONT` 选「自定义」时才用：在 ①-3 勾「字体文件」上传 .ttf／.otf；或上传到左侧「文件」面板（`/content/`）后填路径。请自行确认该字体允许你的用途（例如商用）
SUB_FONT_FILE = "/content/my_font.ttf"  #@param {type:"string"}
# @markdown `SUB_SIZE`：字号（%）：100＝标准大小（画面短边的 1/12，例如 480×832 画面约 40 像素）；字越大，一行放的字越少
SUB_SIZE = 100  #@param {type:"slider", min:50, max:200, step:5}
# @markdown `SUB_EFFECT`：出现效果：淡入＝每段字幕淡淡出现；淡入淡出＝消失时也淡出；弹出＝放大弹出；滑入＝从下往上滑进来；逐字＝按念的速度一个字一个字出现
SUB_EFFECT = "淡入"  #@param ["无", "淡入", "淡入淡出", "弹出", "滑入", "逐字"]
SUB_POSITION = "底部"  #@param ["底部", "中间", "顶部"]
# @markdown `SUB_MARGIN`：字幕离画面下边缘（选「顶部」时是上边缘）的距离，占画面高度的 %；选「中间」时不用。版面选「上主播下录影」时 `SUB_POSITION`、`SUB_MARGIN` 都不用：字幕固定放在上下两格的交界
SUB_MARGIN = 12  #@param {type:"slider", min:0, max:45, step:1}
# @markdown `SUB_OUTLINE`：描边粗细，占字号的 %；0＝不描边（默认单层黑色描边）
SUB_OUTLINE = 8  #@param {type:"slider", min:0, max:20, step:1}
# @markdown `SUB_COLOR`／`SUB_OUTLINE_COLOR`：颜色写成 #RRGGBB，例如白色 #FFFFFF、黄色 #FFE000、黑色 #000000
SUB_COLOR = "#FFFFFF"  #@param {type:"string"}
SUB_OUTLINE_COLOR = "#000000"  #@param {type:"string"}
# @markdown ### 进阶：显存
# @markdown `LOW_VRAM`：对嘴阶段爆显存（OutOfMemory）时勾选再重跑：更慢但更省显存
LOW_VRAM = False  #@param {type:"boolean"}
# @markdown ### 进阶：声音克隆（默认关闭）
# @markdown 只能用你本人、或已取得当事人同意的录音。开启（勾 `ENABLE_VOICE_CLONE`）后 `VOICE` 和 `VOICE_DESC` 都不再使用
ENABLE_VOICE_CLONE = False  #@param {type:"boolean"}
# @markdown `REF_WAV`：参考录音（在 ①-3 勾「参考录音」上传；或上传到 `/content/` 后填路径）；建议 5～15 秒、单人说话、没有背景音乐
REF_WAV = "/content/my_voice.wav"  #@param {type:"string"}
# @markdown `REF_TEXT`：参考录音的逐字稿：填了走「极致克隆」（更像本人，但不能加语气括号）；留空走「可控克隆」
REF_TEXT = ""  #@param {type:"string"}
# @markdown `I_HAVE_CONSENT`：确认录音是你本人、或已取得当事人同意后，才勾选（＝True）
I_HAVE_CONSENT = False  #@param {type:"boolean"}

# ===== 以下是检查和接线，不用改 =====
# 各格共用的状态框（绿色＝完成，黄色＝进行中）
import html as _html
from IPython.display import HTML, display


def show_status(msg, busy=False):
    fg, bg = ("#7a5200", "#fff3cd") if busy else ("#1e6b34", "#e2f3e6")
    display(HTML(f'<div style="border-left:5px solid {fg};background:{bg};color:{fg};padding:6px 12px;'
                 f'margin:6px 0;white-space:pre-wrap">{_html.escape(str(msg))}</div>'))


if ENABLE_VOICE_CLONE and I_HAVE_CONSENT is not True:
    raise PermissionError("已开启声音克隆，但 I_HAVE_CONSENT 不是 True。"
                          "只能用你本人或已取得当事人同意的录音；确认后把 I_HAVE_CONSENT 改成 True 再运行。")
ASPECT_SIZES = {"9:16 竖屏短视频（480×832）": (480, 832), "16:9 横屏视频（832×480）": (832, 480),
                "1:1 方形帖子（624×624）": (624, 624), "4:5 Instagram 竖版帖子（576×720）": (576, 720),
                "3:4 小红书竖版（576×768）": (576, 768)}
if ASPECT not in ASPECT_SIZES:
    raise ValueError(f"不认识的画面比例 {ASPECT!r}：请在 ASPECT 下拉选单里选")
WIDTH, HEIGHT = ASPECT_SIZES[ASPECT]   # 对嘴画面的宽和高（④-3 用）
if CROP_POS not in ("偏上（保留头部）", "置中", "偏左", "偏右"):
    raise ValueError(f"不认识的裁切位置 {CROP_POS!r}：请在 CROP_POS 下拉选单里选")
BG_MODES = {"关": "off", "绿幕（推荐）": "green", "逐帧": "frames"}
if BG_REMOVE not in BG_MODES:
    raise ValueError(f"不认识的去背选项 {BG_REMOVE!r}：请在 BG_REMOVE 下拉选单里选")
BG_MODE = BG_MODES[BG_REMOVE]
if LAYOUT not in ("只有主播", "上主播下录影"):
    raise ValueError(f"不认识的版面 {LAYOUT!r}：请在 LAYOUT 下拉选单里选")
LAYOUT_STREAM = LAYOUT == "上主播下录影"
if LAYOUT_STREAM:
    if BG_MODE != "off":
        raise ValueError("「上主播下录影」用主播的原背景、不去背：请把 BG_REMOVE 改回「关」")
    if not (0 <= REC_CROP_TOP and 0 <= REC_CROP_BOTTOM and REC_CROP_TOP + REC_CROP_BOTTOM < 90):
        raise ValueError("REC_CROP_TOP、REC_CROP_BOTTOM 要是 0 以上，而且加起来小于 90")
    if not 0 <= REC_CENTER_X <= 100:
        raise ValueError(f"REC_CENTER_X 要在 0～100 之间（50＝正中间）；现在是 {REC_CENTER_X}")
    if not 0 <= HOST_CROP_TOP <= 90:
        raise ValueError(f"HOST_CROP_TOP 要在 0～90 之间（%）；现在是 {HOST_CROP_TOP}")
    if not (0 <= BG_DIM <= 1 and 0 <= BG_BLUR):
        raise ValueError("BG_DIM 要在 0～1 之间，BG_BLUR 不能小于 0")

# ①-3 上传的文件：上传记录存在 UPLOAD_RECORD（重启会话也还在，删除运行时就没了），所以重跑本格照样用上传的文件
import json, os

UPLOAD_RECORD = "/content/.talking_avatar_uploads.json"
UPLOAD_VARS = {"IMAGE_PATH": "角色图", "REC_PATH": "录影", "BGM_PATH": "BGM", "SUB_FONT_FILE": "字体文件", "REF_WAV": "参考录音"}
FORM_PATHS = {v: globals()[v] for v in UPLOAD_VARS}   # 本格表单里填的路径（套用上传记录之前的值）


def load_uploads():
    """上传记录：{变量名: {"path": 上传的文件, "form": 上传当时本格表单填的值}}；没有记录回传 {}。"""
    try:
        with open(UPLOAD_RECORD, encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def save_uploads(rec):
    with open(UPLOAD_RECORD, "w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False, indent=1)


def apply_uploads():
    """把上传记录套回路径变量；回传要显示的说明。
    表单值和上传当时一样 → 用上传的文件；表单改成别的值 → 用表单（上传记录作废）；上传的文件不在了 → 用表单。"""
    rec, notes = load_uploads(), []
    for var in [v for v in rec if v in UPLOAD_VARS]:
        r, label = rec[var], UPLOAD_VARS[var]
        if FORM_PATHS[var] != r["form"]:
            notes.append(f"{label}：本格的 {var} 改成了 {FORM_PATHS[var]!r}，改用它（不再用 ①-3 上传的 {r['path']}）")
        elif not os.path.exists(r["path"]):
            notes.append(f"{label}：①-3 上传的 {r['path']} 已经不在，改用本格的 {var}＝{FORM_PATHS[var]!r}")
        else:
            globals()[var] = r["path"]
            notes.append(f"{label}：用 ①-3 上传的 {r['path']}")
            continue
        del rec[var]
        save_uploads(rec)
    return notes


show_status("\n".join([f"✅ 设置已读取：画面 {WIDTH}x{HEIGHT}，裁切「{CROP_POS}」，去背「{BG_REMOVE}」，版面「{LAYOUT}」",
                       ("文稿 %d 行" % sum(1 for s in SCRIPT.splitlines() if s.strip())) if "SCRIPT" in globals()
                       else "文稿：还没运行 ① 文稿格",
                       *apply_uploads()]))

In [ ]:
# @title ①-3 上传文件（选填） { display-mode: "form" }
# @markdown 勾选要上传的项目，运行本格（或「全部运行」）时每一项会出现一个「选择文件」按钮，**每项只选一个文件**；上传后自动把 ①-2 对应的路径指到这个文件。**全部不勾＝什么都不做，直接跳过。**
# @markdown - **上传记录会保留**：之后重跑 ①-2（例如调版面）也照样用上传的文件；上传完可以把勾取消。
# @markdown - **路径规则**：同一项在 ①-2 填了路径、又在这里上传 → 用上传的文件。之后在 ①-2 把那个路径改成别的值 → 改用 ①-2 填的路径（上传记录作废）。勾最下面的「清除上传记录」→ 全部改回 ①-2 填的路径。
# @markdown - 勾着但这次不想换文件：在「选择文件」那里点取消，会沿用之前上传的文件（之前没上传过会报错停下）。
# @markdown - 录影很大（几百 MB）时，用这里上传可能很慢；建议改用左侧「文件」面板上传后，在 ①-2 填 `REC_PATH`。
# @markdown - 跑完一次后才换文件：换角色图、录影、参考录音要「全部运行」；换 BGM 运行本格 → ②-2 → ⑤；换字体运行本格 → ⑥。
# @markdown
# @markdown 角色图（→ `IMAGE_PATH`）
UPLOAD_IMAGE = False  #@param {type:"boolean"}
# @markdown 录影（→ `REC_PATH`，①-2 版面选「上主播下录影」时才用）
UPLOAD_REC = False  #@param {type:"boolean"}
# @markdown 背景音乐（→ `BGM_PATH`，「上主播下录影」时才用）
UPLOAD_BGM = False  #@param {type:"boolean"}
# @markdown 字体文件 .ttf／.otf（→ `SUB_FONT_FILE`，①-2 的 `SUB_FONT` 选「自定义（上传字体文件）」时才用）
UPLOAD_FONT = False  #@param {type:"boolean"}
# @markdown 参考录音（→ `REF_WAV`，①-2 开启声音克隆时才用）
UPLOAD_REF = False  #@param {type:"boolean"}
# @markdown 清除上传记录（全部改回 ①-2 填的路径；用完把勾取消）
UPLOAD_CLEAR = False  #@param {type:"boolean"}

if any(n not in globals() for n in ("show_status", "FORM_PATHS", "apply_uploads")):
    raise RuntimeError("请先运行 ①-2 设置，再运行本格")


def upload_selected():
    """照勾选逐项跳出「选择文件」，把路径变量指到上传的文件并记进上传记录；回传要显示的说明。"""
    import os
    notes = []
    if UPLOAD_CLEAR:
        save_uploads({})
        for var in UPLOAD_VARS:
            globals()[var] = FORM_PATHS[var]
        notes.append("已清除上传记录：全部改回 ①-2 填的路径")
    todo = [var for on, var in ((UPLOAD_IMAGE, "IMAGE_PATH"), (UPLOAD_REC, "REC_PATH"), (UPLOAD_BGM, "BGM_PATH"),
                                (UPLOAD_FONT, "SUB_FONT_FILE"), (UPLOAD_REF, "REF_WAV")) if on]
    if not todo:
        return notes or ["没有勾选要上传的项目，跳过"]
    from google.colab import files
    unused = {"REC_PATH": (not LAYOUT_STREAM, "①-2 的 LAYOUT 不是「上主播下录影」"),
              "BGM_PATH": (not LAYOUT_STREAM, "①-2 的 LAYOUT 不是「上主播下录影」"),
              "SUB_FONT_FILE": (SUB_FONT != "自定义（上传字体文件）", "①-2 的 SUB_FONT 没选「自定义（上传字体文件）」"),
              "REF_WAV": (not ENABLE_VOICE_CLONE, "①-2 没开启声音克隆（ENABLE_VOICE_CLONE）")}
    rec = load_uploads()
    for var in todo:
        label = UPLOAD_VARS[var]
        print(f"请选择「{label}」（只选一个文件）：")
        try:
            got = files.upload()
        except Exception as e:   # 取消或上传中断：可能回传空的，也可能报错，两种都当成没上传
            print(f"（上传没有完成：{e!r}）")
            got = {}
        if len(got) > 1:
            raise ValueError(f"「{label}」一次只能选一个文件，这次选了 {len(got)} 个：{'、'.join(got)}。"
                             f"请重新运行本格，只选一个（这几个文件已经放进 {os.getcwd()}，也可以直接在 ①-2 填路径）")
        if not got:
            if var in rec:
                notes.append(f"{label}：没有选新文件，沿用之前上传的 {rec[var]['path']}")
                continue
            raise RuntimeError(f"「{label}」没有上传（取消了，或没选到文件）。"
                               f"要上传就重新运行本格；不上传就取消勾选「{label}」再运行")
        name, data = next(iter(got.items()))
        path = os.path.join(os.getcwd(), os.path.basename(name))
        with open(path, "wb") as f:   # 自己再写一次：同名文件已经存在时 Colab 可能另存成别的名字，这样路径一定指到这次上传的内容
            f.write(data)
        globals()[var] = path
        rec[var] = {"path": path, "form": FORM_PATHS[var]}
        save_uploads(rec)
        notes.append(f"{label}：已上传 {path}（{len(data) / 1e6:.1f}MB），{var} 已改成这个文件")
        if var in unused and unused[var][0]:
            notes.append(f"  注意：{unused[var][1]}，这个文件现在不会用到")
    return notes


show_status("\n".join(["✅ 上传文件：完成", *upload_selected()]))

In [ ]:
# @title ② 环境检查 { display-mode: "form" }
# @markdown 检查 GPU（应该是 L4）、取得示例文件、确认输入文件（角色图；开了声音克隆时还有参考录音）存在。不用改，几秒就好。
import os, subprocess, sys, time

T_START = time.time()
REPO_URL = "https://github.com/dawang-sakura/colab-talking-avatar"
REPO_DIR = "/content/colab-talking-avatar"
OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)


def run(cmd, **kw):
    """运行外部命令；失败时打印最后 40 行输出并报错。"""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        tail = "\n".join((r.stdout + r.stderr).splitlines()[-40:])
        raise RuntimeError(f"命令失败（exit {r.returncode}）：{' '.join(map(str, cmd))}\n{tail}")
    return r


def gpu_used():
    return subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()


# GPU：本项目只在 L4（约 22.5GB 显存）上实测过
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if not smi:
    raise RuntimeError("没有检测到 GPU：请到「代码执行程序 → 更改运行时类型」选 L4 GPU，保存后重新全部运行")
GPU_NAME, GPU_MIB = [s.strip() for s in smi.splitlines()[0].rsplit(",", 1)]
GPU_MIB = int(GPU_MIB)
print(f"GPU：{GPU_NAME}，显存 {GPU_MIB / 1024:.1f} GiB")
if "L4" not in GPU_NAME or GPU_MIB < 20 * 1024:
    print("⚠ 警告：当前不是 L4，或显存不足 20GB。本项目只在 L4 上实测过，T4（15GB）未测试，"
          "很可能爆显存或非常慢。\n"
          "  请到「代码执行程序 → 更改运行时类型」改成 L4；"
          "浏览器登录了多个 Google 账号时，Colab 默认用第一个账号，该账号没有计算单元会被降成 T4。")

# 取得示例文件（examples/）
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print("⚠ 克隆仓库失败，拿不到 examples/ 示例文件；如果用的是自己上传的图，可以忽略。\n" + r.stderr[-500:])


def resolve(p):
    """相对路径依序在 /content 和仓库目录里找；绝对路径原样返回。"""
    if os.path.isabs(p):
        return p
    for base in ("/content", REPO_DIR):
        if os.path.exists(os.path.join(base, p)):
            return os.path.join(base, p)
    return os.path.join(REPO_DIR, p)


IMAGE_FILE = resolve(IMAGE_PATH)
if not os.path.exists(IMAGE_FILE):
    raise FileNotFoundError(f"找不到角色图 {IMAGE_FILE}：请确认 IMAGE_PATH，或先在左侧「文件」面板上传")
print("角色图：", IMAGE_FILE)

if ENABLE_VOICE_CLONE:
    if I_HAVE_CONSENT is not True:
        raise PermissionError("只能用你本人或已取得当事人同意的录音；确认后把 I_HAVE_CONSENT 改成 True")
    REF_FILE = resolve(REF_WAV)
    if not os.path.exists(REF_FILE):
        raise FileNotFoundError(f"找不到参考录音 {REF_FILE}：请先在左侧「文件」面板上传，并确认 REF_WAV")
    print("参考录音：", REF_FILE)
show_status("✅ 环境检查完成")

In [ ]:
# @title ②-2 版面：检查录影和 BGM { display-mode: "form" }
# @markdown ①-2 的 `LAYOUT` 选「上主播下录影」时：检查录影和 BGM，印出录影要取的范围（对嘴开始之前就能确认裁得对不对），并定义「台词对时」和「上下合成」用的函数。选「只有主播」时只定义函数、不做别的事（③-1 的台词对时两种版面都用；没写秒数时和以前完全一样）。不用改。
# @markdown 跑完后换了 BGM：运行 ①-2（用 ①-3 上传的话再运行 ①-3）→ 本格 → ⑤。
import json, math, os, re, subprocess
import numpy as np

STREAM_W, STREAM_H, STREAM_FPS = 1080, 1920, 25   # 上主播下录影：成品固定 1080×1920（9:16）、25fps
STREAM_TOP_H = 820                                 # 上格（主播）高 820；下格（录影）高 1920−820＝1100；字幕中心放在两格交界 y＝820
TIKTOK_BTN_X, TIKTOK_BTN_Y = 894, 915              # TikTok 右侧按钮列大约从 x≥894、y≥915 开始（字幕要避开）
TIME_RE = re.compile(r"^\s*(\d+(?:\.\d+)?)\s*[｜|]\s*")   # 行首的开始秒数：「14.0｜台词」或「14.0|台词」
ANCHOR_TAIL_SEC = 2.0      # 对嘴只生到最后一句结束后这么多秒（这段是静音，省计算单元）
PINGPONG_SEC = 2.0         # 对嘴片比录影短时，之后的主播画面用最后一句结束后的静音画面（最多这么多秒）来回循环（假装在待机）
VOICE_LUFS = -16.0         # 主播配音的响度目标（最大声）
REC_LUFS, BGM_LUFS = -26.0, -28.0    # 录影原音、BGM 平时的响度目标（比配音小 10～12 LU）
DUCK_DB = -12.0            # 主播讲话时，录影原音和 BGM 再压低这么多 dB
DUCK_ATTACK, DUCK_RELEASE = 0.15, 0.4   # 讲话前 0.15 秒开始压低，讲完后 0.4 秒内恢复


def parse_timed(raw):
    """「14.0｜台词」→ (14.0, "台词")；没写秒数 → (None, 原句)。全角｜和半角 | 都可以。"""
    m = TIME_RE.match(raw)
    if not m:
        return None, raw
    return float(m.group(1)), raw[m.end():]


def schedule_lines(lengths, wants, sr, pause_sec, rec_dur=None):
    """排每句在音轨里的开始位置（样本数）。lengths＝每句的样本数；wants＝每句指定的开始秒数（None＝没指定）。
    没指定：接在上一句后面，隔 pause_sec 秒（第一句从 0 开始）；指定了但上一句还没念完：顺延到上一句结束，并警告。
    rec_dur（录影秒数）不是 None 时，结尾超出录影的句子照放，并警告。回传 (开始位置列表, 警告列表)。"""
    pause = int(pause_sec * sr)
    starts, warns, end = [], [], None
    for i, (n, want) in enumerate(zip(lengths, wants), 1):
        if want is None:
            s = 0 if end is None else end + pause
        else:
            s = round(want * sr)
            if end is not None and s < end:
                warns.append(f"第 {i} 句指定 {want:.2f} 秒开始，但上一句到 {end / sr:.2f} 秒才念完："
                             f"往后顺延到 {end / sr:.2f} 秒（晚 {(end - s) / sr:.2f} 秒）")
                s = end
        starts.append(s)
        end = s + n
        if rec_dur is not None and end / sr > rec_dur + 1e-6:
            warns.append(f"第 {i} 句在 {end / sr:.2f} 秒才念完，超出录影长度 {rec_dur:.2f} 秒："
                         "照放，超出的部分录影停在最后一格")
    return starts, warns


def assemble_voice(wavs, wants, texts, sr, pause_sec, rec_dur=None):
    """把逐句配音按 schedule_lines 排好的位置放进一条音轨。回传 (音轨, 字幕时间轴, 警告列表)。"""
    starts, warns = schedule_lines([len(w) for w in wavs], wants, sr, pause_sec, rec_dur)
    track = np.zeros(max(s + len(w) for s, w in zip(starts, wavs)), dtype=np.float32)
    timeline = []
    for s, w, text in zip(starts, wavs, texts):
        track[s:s + len(w)] = w
        start = s / sr
        timeline.append({"start": round(start, 3), "end": round(start + len(w) / sr, 3), "text": text})
    return track, timeline, warns


def probe_media(path):
    """回传 {dur, has_video, has_audio, w, h, frames}（frames＝视频串流标的帧数，读不到是 0）。"""
    info = json.loads(run(["ffprobe", "-v", "error", "-show_entries",
                           "format=duration:stream=codec_type,width,height,nb_frames", "-of", "json", path]).stdout)
    v = [s for s in info.get("streams", []) if s.get("codec_type") == "video"]
    a = [s for s in info.get("streams", []) if s.get("codec_type") == "audio"]
    nb = v[0].get("nb_frames", "0") if v else "0"
    return {"dur": float(info["format"]["duration"]), "has_video": bool(v), "has_audio": bool(a),
            "w": v[0]["width"] if v else 0, "h": v[0]["height"] if v else 0,
            "frames": int(nb) if str(nb).isdigit() else 0}


def measure_lufs(path, dur=None):
    """整体响度（LUFS，ffmpeg loudnorm 的量测值）；几乎无声时回传 None。"""
    r = run(["ffmpeg", "-hide_banner", "-nostats", "-i", path, *(["-t", f"{dur:.3f}"] if dur else []),
             "-map", "0:a:0", "-af", "loudnorm=print_format=json", "-f", "null", "-"])
    js = r.stderr[r.stderr.rfind("{"):r.stderr.rfind("}") + 1]
    try:
        val = float(json.loads(js)["input_i"])
    except (ValueError, KeyError):
        return None
    return val if val > -70 else None


def _even(x, lo=2):
    """四舍五入到偶数（yuv420 的宽高、裁切位置要是偶数）；lo＝最小值（裁切位置可以是 0）。"""
    return max(lo, int(round(x / 2)) * 2)


def duck_expr(timeline, duck_db=DUCK_DB):
    """主播讲话时段的音量包络（ffmpeg volume 滤镜的算式）：讲话时乘上 duck_db，前后用斜线过渡。"""
    spans = []
    for seg in sorted(timeline, key=lambda s: s["start"]):   # 两句间隔太短时并成一段，免得音量忽高忽低
        s, e = seg["start"] - DUCK_ATTACK, seg["end"] + DUCK_RELEASE
        if spans and s <= spans[-1][1]:
            spans[-1][1] = max(spans[-1][1], e)
        else:
            spans.append([s, e])
    if not spans:
        return "1"
    g = 10 ** (duck_db / 20)
    # 每段：讲话前 DUCK_ATTACK 秒从 0 升到 1、讲完后 DUCK_RELEASE 秒从 1 降回 0；并过的段互不重叠，所以直接相加
    terms = [f"clip(min((t-{s:.3f})/{DUCK_ATTACK},({e:.3f}-t)/{DUCK_RELEASE}),0,1)" for s, e in spans]
    return f"1-{1 - g:.4f}*({'+'.join(terms)})"


def mix_audio(voice_audio, rec_video, out_wav, timeline, dur, rec_audio=True, bgm=None, log=print):
    """混音：主播配音（响度调到 VOICE_LUFS）＋录影原音（REC_LUFS）＋BGM（BGM_LUFS，比成片短就循环），
    主播讲话时录影原音和 BGM 按 timeline 压低 DUCK_DB；最后过限幅器免得爆音。回传量测到的数值。"""
    info = {}
    lv = measure_lufs(voice_audio)
    gv = 0.0 if lv is None else max(-20.0, min(20.0, VOICE_LUFS - lv))
    info["voice_lufs_in"], info["voice_gain_db"] = lv, round(gv, 2)
    inputs = ["-i", voice_audio]
    chains = [f"[0:a]aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo,volume={gv:.2f}dB,"
              f"apad,atrim=0:{dur:.3f}[av]"]
    duck = duck_expr(timeline)
    beds = []
    if rec_audio and probe_media(rec_video)["has_audio"]:
        lr = measure_lufs(rec_video)
        if lr is None:
            log("（录影原音几乎无声，不加）")
        else:
            gr = max(-30.0, min(30.0, REC_LUFS - lr))
            info["rec_lufs_in"], info["rec_gain_db"] = lr, round(gr, 2)
            beds.append(("rec", inputs.count("-i"), gr))
            inputs += ["-i", rec_video]
    if bgm:
        lb = measure_lufs(bgm)
        if lb is None:
            log("（BGM 几乎无声，不加）")
        else:
            gb = max(-30.0, min(30.0, BGM_LUFS - lb))
            info["bgm_lufs_in"], info["bgm_gain_db"] = lb, round(gb, 2)
            beds.append(("bgm", inputs.count("-i"), gb))
            inputs += ["-stream_loop", "-1", "-i", bgm]   # BGM 比成片短时循环播放
    for name, idx, gain in beds:
        # asetnsamples：每 10 毫秒算一次音量，压低／恢复的过渡比较平滑
        chains.append(f"[{idx}:a]aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo,volume={gain:.2f}dB,"
                      f"apad,atrim=0:{dur:.3f},asetnsamples=n=480,volume=eval=frame:volume='{duck}'[a{name}]")
    labels = "[av]" + "".join(f"[a{name}]" for name, _, _ in beds)
    chains.append(f"{labels}amix=inputs={1 + len(beds)}:normalize=0:duration=first,"
                  f"alimiter=limit=0.891:level=disabled[aout]")   # 0.891＝-1 dBFS
    run(["ffmpeg", "-hide_banner", "-v", "error", "-y", *inputs, "-filter_complex", ";".join(chains),
         "-map", "[aout]", "-c:a", "pcm_s16le", "-ar", "48000", out_wav])
    info["beds"] = [name for name, _, _ in beds]
    return info


def rec_layout(rec_w, rec_h, crop_top=0, crop_bottom=0, center_x=50):
    """录影怎么放进下格（1080×1100）。先裁掉上方 crop_top%、下方 crop_bottom%，剩下的高度 ch 整段保留（上下不再裁），
    左右按下格比例取 ch×1080÷1100 宽：
      录影够宽 → 以 center_x%（录影宽度的百分比）为中心取这么宽，超出录影边缘时往内挪（shifted＝True），缩放到 1080×1100；
      录影不够宽 → 整个宽度都用，照高度缩放到 1100 高、左右置中（fw＜1080），两侧留给模糊补边。
    回传 dict：cx, cy, cw, ch＝录影上要取的范围；fw＝放进下格后的宽；shifted＝中心有没有被往内挪。"""
    W, H = STREAM_W, STREAM_H - STREAM_TOP_H
    cy = _even(rec_h * crop_top / 100, lo=0)
    ch = min(_even(rec_h * (100 - crop_top - crop_bottom) / 100), (rec_h - cy) // 2 * 2)
    want_w = ch * W / H
    if want_w <= rec_w:
        cw = min(_even(want_w), rec_w // 2 * 2)
        x0 = round(rec_w * center_x / 100 - cw / 2)
        cx = min(max(0, x0), rec_w - cw) // 2 * 2
        return {"cx": cx, "cy": cy, "cw": cw, "ch": ch, "fw": W, "shifted": not 0 <= x0 <= rec_w - cw}
    return {"cx": 0, "cy": cy, "cw": rec_w // 2 * 2, "ch": ch, "fw": min(W, _even(rec_w * H / ch)), "shifted": False}


def compose_stream(host_video, voice_audio, rec_video, out_mp4, timeline, rec_crop_top=0, rec_crop_bottom=0,
                   rec_center_x=50, host_crop_top=9, rec_audio=True, bgm=None, bg_blur=7, bg_dim=0.6, work_dir=".", log=print):
    """上主播下录影（切成上下两格）：1080×1920＝上格主播 1080×STREAM_TOP_H＋下格录影 1080×(1920−STREAM_TOP_H)，直接上下相接。
    主播（对嘴片）：等比放大到盖满上格（9:16 的主播片＝放大到 1080 宽），左右置中，从上方 host_crop_top% 的高度往下取上格的高。
    录影：照 rec_layout 裁切、缩放；录影比下格窄时，两侧是同一支录影放大盖满下格、模糊（缩到 1/4 尺寸做 gblur，
    bg_blur 是那个尺寸上的 sigma）、亮度乘 bg_dim。
    成片长度＝max(录影, 配音)：对嘴片较短→最后一句结束后的静音画面来回循环；录影较短→停在最后一格。回传量测值。"""
    W, FPS = STREAM_W, STREAM_FPS
    TH, BH = STREAM_TOP_H, STREAM_H - STREAM_TOP_H
    host, rec = probe_media(host_video), probe_media(rec_video)
    if not rec["has_video"]:
        raise ValueError(f"{rec_video} 里没有画面")
    voice_dur = probe_media(voice_audio)["dur"]
    dur = max(rec["dur"], voice_dur)
    # 主播：等比放大到盖满上格（多的部分左右置中裁掉），从上方 host_crop_top% 往下取 TH 高
    zoom = max(W / host["w"], TH / host["h"])
    hw, hh = max(W, _even(host["w"] * zoom)), max(TH, _even(host["h"] * zoom))
    hx = (hw - W) // 4 * 2
    hy_want = _even(hh * host_crop_top / 100, lo=0)
    hy = min(hy_want, hh - TH)
    if hy < hy_want and hh == TH:
        log(f"（主播片放大到盖满上格后刚好 {TH} 像素高，上下没得裁，HOST_CROP_TOP 不起作用）")
    elif hy < hy_want:
        log(f"⚠ HOST_CROP_TOP={host_crop_top}% 太大：主播片放大后只有 {hh} 像素高，上方最多只能裁 {hy / hh * 100:.0f}%，已改用这个值")
    L = rec_layout(rec["w"], rec["h"], rec_crop_top, rec_crop_bottom, rec_center_x)
    if L["shifted"]:
        log(f"⚠ REC_CENTER_X={rec_center_x}：以它为中心取 {L['cw']} 像素宽会超出录影边缘，已往内挪成取 x={L['cx']}–{L['cx'] + L['cw']}"
            f"（中心约 {(L['cx'] + L['cw'] / 2) / rec['w'] * 100:.0f}%）")
    # 主播延长：对嘴片比成片短时，只拿最后一句结束之后的静音画面（最多 PINGPONG_SEC 秒）倒放、正放来回循环，不循环到讲话的画面
    n_host = host["frames"] or round(host["dur"] * FPS)
    short = dur - n_host / FPS
    head = f"[0:v]setpts=PTS-STARTPTS,fps={FPS},format=yuv420p"
    k = 0
    if short > 0.5 / FPS:
        silent_from = math.ceil(max((s["end"] for s in timeline), default=0) * FPS)   # 最后一句结束后的第一帧
        k = min(round(PINGPONG_SEC * FPS), n_host - silent_from)
        if k < 2:   # 对嘴片在最后一句结束后几乎没有画面：只好循环最后几帧（可能带到讲话的画面）
            k = max(2, min(n_host, round(PINGPONG_SEC * FPS)))
            log(f"⚠ 对嘴片在最后一句结束后不到 2 帧，待机循环只能用最后 {k} 帧（可能带到讲话的画面）")
        vf = [f"{head},split=2[hm][ht]",
              f"[ht]trim=start_frame={n_host - k},setpts=PTS-STARTPTS,split=2[tf][tr0]",
              "[tr0]reverse[tr]",
              f"[tr][tf]concat=n=2:v=1:a=0,loop=loop=-1:size={2 * k}:start=0,setpts=N/{FPS}/TB[pp]",
              "[hm][pp]concat=n=2:v=1:a=0[hx]"]
    else:
        vf = [f"{head}[hx]"]
    rec_in = (f"[1:v]setpts=PTS-STARTPTS,fps={FPS},crop={L['cw']}:{L['ch']}:{L['cx']}:{L['cy']},"
              f"tpad=stop_mode=clone:stop_duration={dur:.3f},trim=duration={dur:.3f}")   # 录影较短时停在最后一格
    if L["fw"] >= W:   # 录影够宽：取下来的范围直接缩放成下格
        vf += [f"{rec_in},scale={W}:{BH}:flags=lanczos,setsar=1,format=yuv420p[bot]"]
    else:              # 录影比下格窄：照高度放中间，两侧是同一支录影放大盖满、缩小模糊再放大、压暗
        bw, bh = W // 4, BH // 4 // 2 * 2
        vf += [f"{rec_in},split=2[rb][rf]",
               f"[rb]scale={bw}:{bh}:force_original_aspect_ratio=increase:flags=area,crop={bw}:{bh},gblur=sigma={bg_blur},"
               f"scale={W}:{BH}:flags=bicubic,colorchannelmixer=rr={bg_dim}:gg={bg_dim}:bb={bg_dim},setsar=1[rbg]",
               f"[rf]scale={L['fw']}:{BH}:flags=lanczos,setsar=1[rfg]",
               f"[rbg][rfg]overlay={(W - L['fw']) // 4 * 2}:0,format=yuv420p[bot]"]
    vf += [f"[hx]trim=duration={dur:.3f},setpts=PTS-STARTPTS,scale={hw}:{hh}:flags=lanczos,"
           f"crop={W}:{TH}:{hx}:{hy},setsar=1,format=yuv420p[top]",
           "[top][bot]vstack=inputs=2,format=yuv420p[v]"]
    mixed = os.path.join(work_dir, "layout_mix.wav")
    info = mix_audio(voice_audio, rec_video, mixed, timeline, dur, rec_audio=rec_audio, bgm=bgm, log=log)
    run(["ffmpeg", "-hide_banner", "-v", "error", "-y", "-i", host_video, "-i", rec_video,
         "-i", mixed, "-filter_complex", ";".join(vf), "-map", "[v]", "-map", "2:a",
         "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p", "-r", str(FPS),
         "-c:a", "aac", "-b:a", "192k", "-t", f"{dur:.3f}", "-movflags", "+faststart", out_mp4])
    info.update({"dur": round(dur, 3), "rec_dur": round(rec["dur"], 3), "voice_dur": round(voice_dur, 3),
                 "host_dur": round(n_host / FPS, 3), "pingpong_sec": round(max(0.0, short), 3),
                 "pingpong_frames": [n_host - k, n_host - 1] if k else None,
                 "freeze_sec": round(max(0.0, dur - rec["dur"]), 3),
                 "host_scaled": [hw, hh], "host_crop": [hx, hy, W, TH],                 # 主播放大后的尺寸、取的范围 [x, y, 宽, 高]
                 "rec_crop": [L["cx"], L["cy"], L["cw"], L["ch"]],                    # 录影上取的范围 [x, y, 宽, 高]
                 "rec_box": [(W - L["fw"]) // 4 * 2, TH, (W - L["fw"]) // 4 * 2 + L["fw"], STREAM_H],   # 录影在成品里的位置
                 "rec_side_fill": L["fw"] < W, "rec_center_shifted": L["shifted"]})
    return info


if LAYOUT_STREAM:
    REC_FILE = resolve(REC_PATH)
    if not os.path.exists(REC_FILE):
        raise FileNotFoundError(f"找不到录影 {REC_FILE}：请先在左侧「文件」面板上传，并确认 ①-2 的 REC_PATH")
    try:
        _rec = probe_media(REC_FILE)
    except RuntimeError as e:
        raise ValueError(f"读不了录影 {REC_FILE}（不是视频文件，或格式不支持）：{e}") from None
    if not _rec["has_video"]:
        raise ValueError(f"{REC_FILE} 里没有画面：REC_PATH 要填视频文件")
    REC_DUR = _rec["dur"]
    BGM_FILE = resolve(BGM_PATH) if BGM_PATH.strip() else None
    if BGM_FILE and not os.path.exists(BGM_FILE):
        raise FileNotFoundError(f"找不到 BGM {BGM_FILE}：请先在左侧「文件」面板上传，或把 ①-2 的 BGM_PATH 留空")
    print(f"录影：{REC_FILE}（{_rec['w']}x{_rec['h']}，{REC_DUR:.2f} 秒，{'有' if _rec['has_audio'] else '没有'}声音）")
    _L = rec_layout(_rec["w"], _rec["h"], REC_CROP_TOP, REC_CROP_BOTTOM, REC_CENTER_X)
    print(f"  上方裁掉 {REC_CROP_TOP}%（{_L['cy']} 像素）、下方裁掉 {REC_CROP_BOTTOM}%，剩下 {_L['ch']} 像素高整段放进下格（上下不再裁）")
    if _L["fw"] >= STREAM_W:
        print(f"  左右取 x={_L['cx']}–{_L['cx'] + _L['cw']}（{_L['cw']} 像素宽）→ 缩放到 {STREAM_W}x{STREAM_H - STREAM_TOP_H} 放进下格")
        if _L["shifted"]:
            print(f"⚠ REC_CENTER_X={REC_CENTER_X}：以它为中心取 {_L['cw']} 像素宽会超出录影边缘，已自动往内挪"
                  f"（实际中心约 {(_L['cx'] + _L['cw'] / 2) / _rec['w'] * 100:.0f}%）；游戏不在这个位置的话请调 ①-2 的 REC_CENTER_X")
    else:
        print(f"  录影比下格窄：整个宽度都用，缩放到 {_L['fw']}x{STREAM_H - STREAM_TOP_H} 放在下格中间，"
              "两侧用同一支录影放大、模糊、压暗补满（REC_CENTER_X 不用）")
    if _rec["has_audio"] and not REC_AUDIO:
        print("  录影原音：关（①-2 的 REC_AUDIO = False）")
    print(f"BGM：{BGM_FILE or '不加'}")
    print(f"成品 {STREAM_W}x{STREAM_H}，长度跟着录影走（{REC_DUR:.2f} 秒；配音比录影长时录影停在最后一格）")
else:
    REC_DUR = None
    print("版面：只有主播（和以前一样），本格只定义函数")
show_status(f"✅ ②-2 完成：版面「{LAYOUT}」")

## ③ 配音（VoxCPM2）
- 默认：用 `VOICE` 选的**声音包**参考音（仓库 `voices/`）→ 每句用**可控克隆**生成（同一把声音，句首括号可加语气）。
- `VOICE` 选「自定义（用 VOICE_DESC）」：用 `VOICE_DESC` 做一次**音色设计**，存成参考音，之后同样逐句可控克隆。
- 开启声音克隆：用你的参考录音克隆。`REF_TEXT` 留空＝可控克隆（可加语气）；填了逐字稿＝极致克隆（更像本人，**不能**加语气括号）。
- 句子之间插入 `PAUSE_SEC` 秒停顿，接成一条音轨 `/content/output/voice.wav`。
- 同时记下每句在音轨里的开始／结束时间（字幕时间轴），存成 `/content/output/subtitle_timeline.json`。
- 跑完会删除模型并释放显存，再进入对嘴。生成耗时大约等于音频长度。
- ③-2 字幕准备：下载 ①-2 选的字体、检查字体缺不缺字。字体有问题会在这里先报错，不用等对嘴跑完才发现。

In [ ]:
# @title ③-1 配音 { display-mode: "form" }
# @markdown 安装 VoxCPM2 → 音色设计（或用声音包／参考录音）→ 逐句生成 → 接成一条音轨（同时记下字幕时间轴）→ 释放显存。不用改。
# @markdown 作者实测：开机、安装并载入 VoxCPM2 约 3～4 分钟；之后逐句生成，耗时约等于配音长度。跑完会出现可以试听的音频条。
import gc, json, re

VOXCPM_PIP = "git+https://github.com/OpenBMB/VoxCPM.git@cce58a5b59303c9bd63d12f23afe6a49a4a80c59"  # PyPI 版的 generate() 没有 seed 参数，固定这个 commit
DESIGN_TEXT = "大家好，欢迎收看今天的节目，我们马上开始。"   # 音色设计时念的参考句（只用来定声音，不会出现在成品里）
MAX_CHARS = 80
TONE_RE = re.compile(r"^\s*[(（]([^()（）]*)[)）]\s*")


def split_long(text, limit=MAX_CHARS):
    """超过 limit 字时优先在标点处切开，找不到标点就硬切。"""
    parts = []
    while len(text) > limit:
        cut = max(text.rfind(p, 0, limit) for p in "。！？；，、!?;,")
        cut = cut + 1 if cut > 0 else limit
        parts.append(text[:cut].strip())
        text = text[cut:].strip()
    if text:
        parts.append(text)
    return parts


# 解析文稿：[(指定开始秒数 或 None, 语气 或 None, 句子)]；行首的秒数只给该行切出来的第一段，句首括号的语气套用到每一段
LINES = []
for raw in SCRIPT.strip().splitlines():
    raw = raw.strip()
    if not raw:
        continue
    want, raw = parse_timed(raw)   # 「14.0｜台词」→ 14.0（②-2 定义）
    m = TONE_RE.match(raw)
    tone = (m.group(1).strip() or None) if m else None
    body = raw[m.end():] if m else raw
    for k, part in enumerate(split_long(body)):
        LINES.append((want if k == 0 else None, tone, part))
if not LINES:
    raise ValueError("SCRIPT 是空的：请在 ① 文稿格里填写文稿")
EXTREME = ENABLE_VOICE_CLONE and REF_TEXT.strip() != ""
if EXTREME and any(t for _, t, _ in LINES):
    raise ValueError("极致克隆（填了 REF_TEXT）不能和语气括号一起用：请去掉句首括号，或把 REF_TEXT 留空改走可控克隆")
print(f"共 {len(LINES)} 句：")
for i, (w, t, s) in enumerate(LINES, 1):
    print(f"  {i:02d} {f'[{w:g} 秒] ' if w is not None else ''}{'(' + t + ')' if t else ''}{s}")

# 声音包：没开声音克隆、VOICE 也不是「自定义」时，用仓库 voices/ 里的参考音（先检查，免得装完模型才报错）
USE_PACK = not ENABLE_VOICE_CLONE and VOICE != "自定义（用 VOICE_DESC）"
if USE_PACK:
    pack_json = os.path.join(REPO_DIR, "voices", "voices.json")
    hit = []
    if os.path.exists(pack_json):
        with open(pack_json, encoding="utf-8") as f:
            hit = [v for v in json.load(f) if v["name"] == VOICE]
    PACK_WAV = os.path.join(REPO_DIR, "voices", hit[0]["file"]) if hit else ""
    if not PACK_WAV or not os.path.exists(PACK_WAV):
        raise FileNotFoundError(f"找不到声音包「{VOICE}」的参考音（应该在 {REPO_DIR}/voices/ 里）。"
                                "多半是 ② 克隆仓库失败：请重新运行 ② 再运行 ③，"
                                "或在 ①-2 把 VOICE 改成「自定义（用 VOICE_DESC）」")

show_status("⏳ 进行中，约 3～4 分钟（作者实测，含开机）：安装并载入 VoxCPM2，之后逐句生成，耗时约等于配音长度", busy=True)
t0 = time.time()
run([sys.executable, "-m", "pip", "install", "-q", VOXCPM_PIP])   # 会提示 gcsfs/fsspec 版本冲突，不影响
import numpy as np
import soundfile as sf
import torch
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SR = model.tts_model.sample_rate
print(f"VoxCPM2 就绪（采样率 {SR}），安装＋载入 {time.time() - t0:.0f} 秒")


def synth(text, **kw):
    return model.generate(text=text, cfg_value=2.0, inference_timesteps=10, seed=SEED, **kw)


if ENABLE_VOICE_CLONE:
    ref_path = f"{OUT_DIR}/ref_clone.wav"
    run(["ffmpeg", "-v", "error", "-y", "-i", REF_FILE, "-ac", "1", ref_path])   # 统一转成单声道 wav
    clone_kw = dict(reference_wav_path=ref_path)
    if EXTREME:
        # 极致克隆：prompt_wav_path + prompt_text（逐字稿必须和录音一字不差）
        clone_kw.update(prompt_wav_path=ref_path, prompt_text=REF_TEXT.strip())
    print("声音来源：", "极致克隆" if EXTREME else "可控克隆", REF_FILE)
elif USE_PACK:
    clone_kw = dict(reference_wav_path=PACK_WAV)
    print(f"声音来源：声音包「{VOICE}」，参考音 {PACK_WAV}")
else:
    # 音色设计：描述写在 text 开头的括号里；每次结果不同，所以只做一次并存成参考音，之后都用克隆
    wav = synth(f"({VOICE_DESC}){DESIGN_TEXT}")
    ref_path = f"{OUT_DIR}/voice_design_ref.wav"
    sf.write(ref_path, wav, SR)
    clone_kw = dict(reference_wav_path=ref_path)
    print(f"音色设计完成：{ref_path}（{len(wav) / SR:.2f} 秒）")

WAVS = []
for i, (_, tone, text) in enumerate(LINES, 1):
    t1 = time.time()
    wav = np.asarray(synth(f"({tone}){text}" if tone else text, **clone_kw), dtype=np.float32).reshape(-1)
    WAVS.append(wav)
    print(f"  {i:02d} 生成 {len(wav) / SR:.2f} 秒（耗时 {time.time() - t1:.0f} 秒）")
# 排时间（②-2 的 assemble_voice）：没写秒数的句子接在上一句后面、隔 PAUSE_SEC 秒；写了秒数的从那一秒开始
# SUB_TIMELINE＝字幕时间轴：每句在音轨里的开始／结束秒数＋文稿原文（不含语气括号和秒数）
voice, SUB_TIMELINE, _warns = assemble_voice(WAVS, [w for w, _, _ in LINES], [s for _, _, s in LINES],
                                             SR, PAUSE_SEC, REC_DUR if LAYOUT_STREAM else None)
for _w in _warns:
    print("⚠ " + _w)
if any(w is not None for w, _, _ in LINES):
    print("每句的实际时段：" + "；".join(f"{k}) {seg['start']:.2f}–{seg['end']:.2f}" for k, seg in enumerate(SUB_TIMELINE, 1)))
VOICE_WAV = f"{OUT_DIR}/voice.wav"
sf.write(VOICE_WAV, voice, SR)
SUB_TIMELINE_JSON = f"{OUT_DIR}/subtitle_timeline.json"   # 字幕关闭也照样存，之后可以用 ⑥ 补上字幕
with open(SUB_TIMELINE_JSON, "w", encoding="utf-8") as f:
    json.dump(SUB_TIMELINE, f, ensure_ascii=False, indent=1)
T_VOICE = time.time() - t0
print(f"配音完成：{VOICE_WAV}，共 {len(voice) / SR:.2f} 秒；本步耗时 {T_VOICE:.0f} 秒")

# 释放显存，留给对嘴阶段
del model
gc.collect()
torch.cuda.empty_cache()
print("已释放 VoxCPM2 显存，目前显存占用：", gpu_used())

from IPython.display import Audio, display
display(Audio(VOICE_WAV))
show_status(f"✅ 配音完成：{len(voice) / SR:.2f} 秒，本步耗时 {T_VOICE:.0f} 秒；可以在上面试听")

In [ ]:
# @title ③-2 字幕准备 { display-mode: "form" }
# @markdown 检查 ffmpeg、下载 ①-2 选的字体、检查字体缺不缺字（有问题在这里就报错，不用等对嘴跑完才发现）。本格同时定义 ⑤、⑥ 要用的字幕函数；真正把字幕烧进画面在 ⑤，换样式重做用 ⑥。不用改，很快。
import json, math, os, re, shutil, subprocess, sys, time, unicodedata, urllib.request, warnings, zipfile

SUB_TIMELINE_JSON = f"{OUT_DIR}/subtitle_timeline.json"   # ③-1 记下的字幕时间轴
CLEAN_MP4 = f"{OUT_DIR}/talking_avatar_nosub.mp4"         # ⑤ 合成的不带字幕版本（⑥ 拿它重烧）
OUT_MP4 = f"{OUT_DIR}/talking_avatar.mp4"                 # 成品（字幕开启时带字幕）
SUB_ASS = f"{OUT_DIR}/talking_avatar.ass"                 # 烧进画面用的字幕：ASS 格式，样式和效果都写在里面
SUB_SRT = f"{OUT_DIR}/talking_avatar.srt"                 # 另外给你的纯文字字幕（可导入剪辑软件或上传到平台）
FONT_DIR = "/content/fonts"
CUSTOM_FONT = "自定义（上传字体文件）"

# 可选字体：全部是开源字体，只从官方发布处下载（授权见 README「字幕」；IPA＝IPA Font License 1.0，其余是 SIL OFL 1.1）
#   名称: (下载网址, zip 里要取的文件, 授权, 发布处)
#   「zip 里要取的文件」是 None＝网址直接就是字体文件；写了路径＝网址是 zip，下载后只取出这个文件，zip 随后删掉
SUB_FONTS = {
    "思源黑体（粗）": ("https://github.com/adobe-fonts/source-han-sans/raw/release/OTF/SimplifiedChinese/SourceHanSansSC-Bold.otf",
                None, "SIL OFL 1.1", "Adobe 官方 GitHub adobe-fonts/source-han-sans"),
    "站酷快乐体": ("https://github.com/google/fonts/raw/main/ofl/zcoolkuaile/ZCOOLKuaiLe-Regular.ttf",
              None, "SIL OFL 1.1", "Google Fonts 官方 GitHub google/fonts"),
    "未来荧黑 Heavy": ("https://github.com/welai/glow-sans/releases/download/v0.93/GlowSansSC-Normal-v0.93.zip",
                  "GlowSansSC-Normal-Heavy.otf", "SIL OFL 1.1", "作者官方 GitHub Releases welai/glow-sans（压缩包约 64MB）"),
    "得意黑": ("https://github.com/atelier-anchor/smiley-sans/releases/download/v2.0.1/smiley-sans-v2.0.1.zip",
            "SmileySans-Oblique.ttf", "SIL OFL 1.1", "作者官方 GitHub Releases atelier-anchor/smiley-sans"),
    "寒蝉圆黑体": ("https://github.com/Warren2060/ChillRound/releases/download/v1.805/ChillRoundM_v1.805.zip",
              "ChillRoundM_v1.805/ChillRoundM.ttf", "SIL OFL 1.1", "作者官方 GitHub Releases Warren2060/ChillRound"),
    "霞鹜新晰黑": ("https://github.com/lxgw/LxgwNeoXiHei/releases/download/v1.308/LXGWNeoXiHei.ttf",
              None, "IPA Font License 1.0", "作者官方 GitHub Releases lxgw/LxgwNeoXiHei"),
    "霞鹜文楷": ("https://github.com/lxgw/LxgwWenKai/releases/download/v1.530/LXGWWenKai-Medium.ttf",
             None, "SIL OFL 1.1", "作者官方 GitHub Releases lxgw/LxgwWenKai"),
}
FONT_MAGIC = (b"\x00\x01\x00\x00", b"OTTO", b"true", b"ttcf")   # .ttf／.otf／.ttc 文件开头的 4 个字节
ALIGN = {"底部": 2, "中间": 5, "顶部": 8}                        # ASS 对齐代号：底部居中／正中间／顶部居中
EFFECTS = ("无", "淡入", "淡入淡出", "弹出", "滑入", "逐字")
BREAK_AFTER = "，。！？；：、,.!?;:）)」』”》…"                  # 一行放不下时，优先在这些标点后面切
NO_LINE_START = "，。！？；：、,.!?;:）)」』”’》…—"             # 这些标点不放在一行的开头
JIEBA_PIN = "jieba==0.42.1"   # 中文分词（MIT 授权）：一行放不下要切开时，只在词和词之间切；装不上就退回按字切
_JIEBA = None                 # 分词器只载入一次（False＝载入失败，不再重试）
NO_LINE_START_WORDS = set("了的着过吗呢吧啊呀啦嘛")   # 分词后单独成词的这些助词跟着前一个词，不放在一行的开头
COMMA_PERIOD_RE = re.compile(r"[，。]|(?<!\d)[,.]|[,.](?!\d)")   # 逗号、句号（半角 , . 夹在两个数字中间是小数点／千分位，不算）


def check_ffmpeg():
    """烧字幕要 ffmpeg 的 ass 字幕滤镜（libass）和 libx264 编码器。"""
    if not re.search(r"^\s*\S+\s+ass\s", run(["ffmpeg", "-hide_banner", "-filters"]).stdout, re.M):
        raise RuntimeError("这台机器的 ffmpeg 没有 ass 字幕滤镜（libass），烧不了字幕。"
                           "可以在 ①-2 把 SUBTITLE 改成 False，只出不带字幕的成品")
    if "libx264" not in run(["ffmpeg", "-hide_banner", "-encoders"]).stdout:
        raise RuntimeError("这台机器的 ffmpeg 没有 libx264 编码器，烧不了字幕。"
                           "可以在 ①-2 把 SUBTITLE 改成 False，只出不带字幕的成品")


def ass_color(rgb, name):
    """#RRGGBB → ASS 的 &H00BBGGRR（ASS 颜色是蓝绿红倒过来写，前面的 00＝不透明）。"""
    h = str(rgb).strip().lstrip("#")
    if not re.fullmatch(r"[0-9A-Fa-f]{6}", h):
        raise ValueError(f"{name} 要写成 #RRGGBB，例如白色 #FFFFFF；现在是 {rgb!r}")
    return "&H00" + (h[4:6] + h[2:4] + h[0:2]).upper()


def _is_font(path):
    with open(path, "rb") as f:
        return f.read(4) in FONT_MAGIC


def prepare_font():
    """准备 ①-2 选的字体；回传 (字体文件夹, 字体文件)。每种字体单独一个文件夹，烧字幕时只让 ffmpeg 看到这一个。"""
    if SUB_FONT == CUSTOM_FONT:
        src = resolve(SUB_FONT_FILE)
        if not os.path.exists(src):
            raise FileNotFoundError(f"找不到自定义字体 {src}：请先上传到左侧「文件」面板，再确认 ①-2 的 SUB_FONT_FILE")
        if not _is_font(src):
            raise ValueError(f"{src} 不是 .ttf／.otf／.ttc 字体文件")
        d = f"{FONT_DIR}/custom"
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d)
        return d, shutil.copy(src, d)
    if SUB_FONT not in SUB_FONTS:
        raise ValueError(f"不认识的字体 {SUB_FONT!r}：请在 ①-2 的 SUB_FONT 下拉选单里选")
    url, member, lic, origin = SUB_FONTS[SUB_FONT]
    name = os.path.basename(member or url)
    d = f"{FONT_DIR}/{os.path.splitext(name)[0]}"
    path = f"{d}/{name}"
    if not os.path.exists(path):
        os.makedirs(d, exist_ok=True)
        print(f"下载字体「{SUB_FONT}」（{lic}，来源：{origin}）……")
        hint = f"可以在 ①-2 换一个字体，或自己下载字体文件后选「{CUSTOM_FONT}」"
        dl = f"{d}/download.zip" if member else path + ".part"   # 网址是 zip 时，先存成 zip，取出需要的文件后删掉
        try:
            with urllib.request.urlopen(url, timeout=300) as r, open(dl, "wb") as f:
                shutil.copyfileobj(r, f)
        except Exception as e:
            if os.path.exists(dl):
                os.remove(dl)
            raise RuntimeError(f"字体「{SUB_FONT}」下载失败：{e}\n网址：{url}\n{hint}") from None
        if member:
            try:
                if not zipfile.is_zipfile(dl):
                    raise RuntimeError(f"下载到的不是 zip 压缩包（网址可能已经失效）：{url}\n{hint}")
                with zipfile.ZipFile(dl) as zf:
                    if member not in zf.namelist():
                        raise RuntimeError(f"压缩包里找不到 {member}（作者可能改了包内结构）：{url}\n{hint}")
                    with zf.open(member) as zsrc, open(path + ".part", "wb") as f:   # 只取这一个文件，直接写成 path，不带包内文件夹
                        shutil.copyfileobj(zsrc, f)
            finally:
                os.remove(dl)   # zip 用完就删，不占 Colab 磁盘
        if not _is_font(path + ".part"):
            os.remove(path + ".part")
            raise RuntimeError(f"下载到的不是字体文件（网址可能已经失效）：{url}\n{hint}")
        os.replace(path + ".part", path)
        print(f"  下载完成，{os.path.getsize(path) / 1e6:.1f}MB")
    return d, path


def font_meta(path):
    """读字体内部的家族名（ASS 靠它找字体）、是否粗体、收录了哪些字（None＝没办法检查）。"""
    chars = None
    try:
        from fontTools.ttLib import TTFont
        ft = TTFont(path, fontNumber=0, lazy=True)
        family = ft["name"].getDebugName(1)
        bold = "OS/2" in ft and ft["OS/2"].usWeightClass >= 600
        chars = set(ft.getBestCmap() or {})
        ft.close()
        if family:
            return family, bold, chars
    except ImportError:
        pass
    from PIL import ImageFont   # 没有 fontTools、或读不到家族名时，改用 PIL 读
    family, style = ImageFont.truetype(path, 20).getname()
    return family, "bold" in (style or "").lower(), chars


def load_timeline():
    if not os.path.exists(SUB_TIMELINE_JSON):
        raise FileNotFoundError(f"找不到字幕时间轴 {SUB_TIMELINE_JSON}：要先跑完 ③-1 配音。"
                                "删除过运行时的话文件已经没了，只能重新全部运行")
    with open(SUB_TIMELINE_JSON, encoding="utf-8") as f:
        return json.load(f)


def prepare_subtitles():
    """检查设置和 ffmpeg、准备字体、检查缺字；回传 (时间轴, 字体文件夹, 字体家族名, 是否粗体)。"""
    if SUB_EFFECT not in EFFECTS:
        raise ValueError(f"SUB_EFFECT 只能是 {'、'.join(EFFECTS)}；现在是 {SUB_EFFECT!r}")
    if SUB_POSITION not in ALIGN:
        raise ValueError(f"SUB_POSITION 只能是 底部、中间、顶部；现在是 {SUB_POSITION!r}")
    ass_color(SUB_COLOR, "SUB_COLOR")
    ass_color(SUB_OUTLINE_COLOR, "SUB_OUTLINE_COLOR")
    check_ffmpeg()
    timeline = load_timeline()
    font_dir, font_path = prepare_font()
    family, bold, chars = font_meta(font_path)
    if chars is None:
        print("（没有 fontTools，跳过缺字检查）")
    else:
        miss = sorted({c for seg in timeline for c in seg["text"] if not c.isspace() and ord(c) not in chars})
        if miss:
            print(f"⚠ 字体「{SUB_FONT}」缺少这些字：{''.join(miss)}\n"
                  "  烧进画面时这些字会被换成别的字体，或显示成方框；介意的话在 ①-2 换字体（思源黑体收的字最全）")
    return timeline, font_dir, family, bold


def _char_w(c):
    """估算字宽（以字号为 1）：中文等全角字算 1，英文、数字等半角字约 0.55。"""
    return 1.0 if unicodedata.east_asian_width(c) in "WFA" else 0.55


def _spoken(s):
    """念出来的字数（不算空格和标点）；切段后按它分配时间。"""
    return sum(1 for c in s if not c.isspace() and not unicodedata.category(c).startswith("P"))


def _word_breaks(s):
    """s 里词和词之间的位置（用 jieba 分词）；回传 None＝分词不可用，哪里都能切。"""
    global _JIEBA
    if _JIEBA is None:
        try:
            # jieba 的源码在新版 Python（3.12 起）载入时会印 SyntaxWarning（无害）；只在载入 jieba 这一段压掉，其他警告照常显示
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", SyntaxWarning)
                try:
                    import jieba
                except ImportError:
                    subprocess.run([sys.executable, "-m", "pip", "install", "-q", JIEBA_PIN],
                                   check=True, capture_output=True, timeout=300)
                    import jieba
                jieba.setLogLevel(60)   # 不打印载入词典的讯息
                jieba.initialize()
            _JIEBA = jieba
        except Exception as e:
            _JIEBA = False
            print(f"（中文分词 jieba 载入失败，字幕改成按字切段，可能会切在词中间：{e!r}）")
    if not _JIEBA:
        return None
    pos, breaks = 0, set()
    for w in _JIEBA.cut(s):
        if w in NO_LINE_START_WORDS:
            breaks.discard(pos)
        pos += len(w)
        breaks.add(pos)
    return breaks


def wrap_text(text, max_w):
    """一行放不下时切成几段：尽量等长、优先在标点后面切、只在词和词之间切（不拆开中文词和英文单词）、标点不放行首。"""
    parts, rest = [], text.strip()
    while sum(map(_char_w, rest)) > max_w:
        total = sum(map(_char_w, rest))
        target = total / math.ceil(total / max_w)
        breaks = _word_breaks(rest)
        best, best_score, acc = 1, None, 0.0
        for i in range(1, len(rest)):
            acc += _char_w(rest[i - 1])
            if acc > max_w:
                break
            score = abs(acc - target)
            if rest[i - 1] in BREAK_AFTER:
                score -= max_w * 0.3
            if rest[i - 1].isascii() and rest[i - 1].isalnum() and rest[i].isascii() and rest[i].isalnum():
                score += max_w
            if breaks is not None and i not in breaks:   # 切在词中间：除非一整行内找不到词和词之间的位置，否则不选
                score += max_w
            if rest[i] in NO_LINE_START:
                score += max_w * 2
            if best_score is None or score < best_score:
                best, best_score = i, score
        parts.append(rest[:best].strip())
        rest = rest[best:].strip()
    if rest:
        parts.append(rest)
    return parts


def drop_comma_period(s):
    """①-2 的 SUB_NO_COMMA_PERIOD：句尾的 ，。,. 删掉，句中的换成一个空格（连着好几个也只留一个空格）；其他标点照留。"""
    return re.sub(r"\s+", " ", COMMA_PERIOD_RE.sub(" ", s)).strip()


def build_events(timeline, max_w):
    """每句切成一行一段，按字数比例分时间；回传 [显示开始, 显示结束, 这段念完的时间, 文字]（秒）。"""
    events = []
    for seg in timeline:
        parts = wrap_text(seg["text"], max_w)   # 先按原文切段（优先在标点后切），再去逗号句号
        if SUB_NO_COMMA_PERIOD:
            parts = [p for p in map(drop_comma_period, parts) if p]
            if not parts:
                continue
        weights = [max(1, _spoken(p)) for p in parts]
        t, dur = seg["start"], seg["end"] - seg["start"]
        for p, w in zip(parts, weights):
            t_end = t + dur * w / sum(weights)
            events.append([t, t_end, t_end, p])
            t = t_end
    # 句间停顿时让上一段字幕继续留着，直到下一段出现（避免字幕一闪一闪）；
    # 但最多多留 max(PAUSE_SEC, 1) 秒：行首写了开始秒数、两句隔很久时，不让上一句一直挂着
    hold = max(globals().get("PAUSE_SEC", 0.4), 1.0) + 0.05
    for a, b in zip(events, events[1:]):
        a[1] = min(b[0], a[2] + hold)
    return events


def _ass_t(s):
    cs = max(0, round(s * 100))
    return f"{cs // 360000}:{cs // 6000 % 60:02d}:{cs // 100 % 60:02d}.{cs % 100:02d}"


def _srt_t(s):
    ms = max(0, round(s * 1000))
    return f"{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}"


def _esc(s):
    """ASS 里大括号和反斜线是指令符号，换成全角，免得字幕文字被当成指令。"""
    return s.replace("\\", "＼").replace("{", "｛").replace("}", "｝")


def effect_text(text, show_ms, speech_ms, x, y, font_px):
    """按 SUB_EFFECT 给这一段字幕加 ASS 特效标签。show_ms＝显示多久；speech_ms＝这段念多久（逐字用）；x, y＝定位点（滑入用）。"""
    t = _esc(text)
    fade = max(1, min(200, show_ms // 3))
    if SUB_EFFECT == "淡入":
        return rf"{{\fad({fade},0)}}" + t
    if SUB_EFFECT == "淡入淡出":
        return rf"{{\fad({fade},{fade})}}" + t
    if SUB_EFFECT == "弹出":   # 从 60% 放大到 110%，再回到 100%
        p1, p2 = max(1, min(120, show_ms // 4)), max(2, min(200, show_ms // 3))
        return rf"{{\fscx60\fscy60\t(0,{p1},\fscx110\fscy110)\t({p1},{p2},\fscx100\fscy100)}}" + t
    if SUB_EFFECT == "滑入":   # 从下方 0.6 个字高的地方往上滑到定位点，同时淡入
        s, dy = max(1, min(250, show_ms // 3)), round(font_px * 0.6)
        return rf"{{\move({x},{y + dy},{x},{y},0,{s})\fad({s},0)}}" + t
    if SUB_EFFECT == "逐字":   # 每个字先全透明，念到它的时间点在 80 毫秒内变成不透明；标点、空格跟着前一个字出现
        n, k, out = max(1, _spoken(text)), 0, []
        for c in text:
            if c.isspace() or unicodedata.category(c).startswith("P"):
                t0 = round(speech_ms * max(0, k - 1) / n)
            else:
                t0 = round(speech_ms * k / n)
                k += 1
            out.append(rf"{{\alpha&HFF&\t({t0},{t0 + 80},\alpha&H00&)}}" + _esc(c))
        return "".join(out)
    return t   # 无


def write_subtitles(timeline, family, bold, W, H, center_y=None):
    """按画面尺寸 W×H 生成 ASS（烧进画面用）和 SRT（另外给的）；回传字幕段数。
    center_y 不是 None（上主播下录影）时：字幕中心固定放在这个高度，不用 SUB_POSITION、SUB_MARGIN。"""
    font_px = max(8, round(min(W, H) / 12 * SUB_SIZE / 100))   # 字号：画面短边的 1/12 × SUB_SIZE%
    outline_px = round(font_px * SUB_OUTLINE / 100, 1)          # 描边：字号的 SUB_OUTLINE%
    margin_v = round(H * SUB_MARGIN / 100)                      # 上下边距：画面高度的 SUB_MARGIN%
    margin_lr = max(8, round(W * 0.06))                         # 左右各留画面宽度的 6%
    max_w = (W - 2 * margin_lr - 2 * outline_px) / font_px      # 一行最多放几个中文字宽
    events = build_events(timeline, max_w)
    pos = ""
    if center_y is None:
        align = ALIGN[SUB_POSITION]
        x, y = W // 2, {2: H - margin_v, 5: H // 2, 8: margin_v}[align]   # 滑入要用的定位点
    else:
        # 字幕下缘要在 TikTok 右侧按钮列（y≥TIKTOK_BTN_Y）上方：字大时整行往上收
        align, x = 5, W // 2
        y = min(round(center_y), round(TIKTOK_BTN_Y - font_px * 0.6 - outline_px - 4))
        if SUB_EFFECT != "滑入":   # 滑入自己用 \move 定位
            pos = rf"{{\pos({x},{y})}}"
    ass = [
        "[Script Info]", "ScriptType: v4.00+", f"PlayResX: {W}", f"PlayResY: {H}",
        "WrapStyle: 2",   # 不自动换行：长句已经切成一行一段
        "ScaledBorderAndShadow: yes", "YCbCr Matrix: None", "",
        "[V4+ Styles]",
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, "
        "Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, "
        "MarginL, MarginR, MarginV, Encoding",
        f"Style: Default,{family},{font_px},{ass_color(SUB_COLOR, 'SUB_COLOR')},&H000000FF,"
        f"{ass_color(SUB_OUTLINE_COLOR, 'SUB_OUTLINE_COLOR')},&H00000000,{-1 if bold else 0},0,0,0,"
        f"100,100,0,0,1,{outline_px},0,{align},{margin_lr},{margin_lr},{margin_v},1",
        "",
        "[Events]",
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text",
    ]
    srt = []
    for i, (s, e, se, text) in enumerate(events, 1):
        show_ms, speech_ms = round((e - s) * 1000), round((se - s) * 1000)
        ass.append(f"Dialogue: 0,{_ass_t(s)},{_ass_t(e)},Default,,0,0,0,,"
                   f"{pos}{effect_text(text, show_ms, speech_ms, x, y, font_px)}")
        srt.append(f"{i}\n{_srt_t(s)} --> {_srt_t(e)}\n{text}\n")
    with open(SUB_ASS, "w", encoding="utf-8") as f:
        f.write("\n".join(ass) + "\n")
    with open(SUB_SRT, "w", encoding="utf-8") as f:
        f.write("\n".join(srt))
    print(f"  画面 {W}x{H}：字号 {font_px} 像素、描边 {outline_px} 像素，一行最多约 {max_w:.1f} 个中文字")
    return len(events)


def video_size():
    """画面尺寸一律从不带字幕的版本读回来：重启过会话、或对嘴之后又改了 ①-2 的 ASPECT，都按视频的实际尺寸做字幕。"""
    wh = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height",
              "-of", "csv=p=0:s=x", CLEAN_MP4]).stdout.strip()
    w, h = (int(v) for v in wh.split("x"))
    expect = (STREAM_W, STREAM_H) if globals().get("LAYOUT_STREAM") else (globals().get("WIDTH"), globals().get("HEIGHT"))
    if expect[0] and (w, h) != expect:
        print(f"（视频实际是 {w}x{h}，和 ①-2 现在的设置 {expect[0]}x{expect[1]} 不同：字幕按视频实际尺寸做；"
              "要换画面比例或版面得重新全部运行）")
    return w, h


def sub_signature():
    """现在的字幕设置＋素材文件的修改时间；和上次做成品时一样，⑥ 就不重做。"""
    paths = [CLEAN_MP4, SUB_TIMELINE_JSON] + ([resolve(SUB_FONT_FILE)] if SUB_FONT == CUSTOM_FONT else [])
    return (SUBTITLE, SUB_NO_COMMA_PERIOD, SUB_FONT, SUB_FONT_FILE, SUB_SIZE, SUB_EFFECT, SUB_POSITION, SUB_MARGIN,
            SUB_OUTLINE, SUB_COLOR, SUB_OUTLINE_COLOR, tuple(os.path.getmtime(p) if os.path.exists(p) else 0 for p in paths))


def make_final_video():
    """按 ①-2 现在的字幕设置，从不带字幕的版本做出成品 OUT_MP4；回传要下载的文件。⑤ 和 ⑥ 都用它。"""
    global SUB_LAST_SIG
    if not os.path.exists(CLEAN_MP4):
        raise FileNotFoundError(f"找不到不带字幕的版本 {CLEAN_MP4}：要先完整跑完一次（到 ⑤）。"
                                "删除过运行时的话文件已经没了，只能重新全部运行")
    if not SUBTITLE:
        shutil.copyfile(CLEAN_MP4, OUT_MP4)
        SUB_LAST_SIG = sub_signature()
        print("字幕：关闭（①-2 的 SUBTITLE = False），成品不带字幕")
        return [OUT_MP4]
    timeline, font_dir, family, bold = prepare_subtitles()
    W, H = video_size()
    center_y = None
    if globals().get("LAYOUT_STREAM"):
        center_y = STREAM_TOP_H   # 上主播下录影：字幕中心放在两格交界（②-2 的 STREAM_TOP_H）
        print(f"（上主播下录影：字幕中心放在两格交界 y={center_y}，不用 SUB_POSITION、SUB_MARGIN；字大时会往上收，避开 TikTok 右侧按钮列）")
    n = write_subtitles(timeline, family, bold, W, H, center_y)
    t0 = time.time()
    try:
        r = run(["ffmpeg", "-hide_banner", "-nostats", "-y", "-i", CLEAN_MP4,
                 "-vf", f"ass={SUB_ASS}:fontsdir={font_dir}",
                 "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p",
                 "-c:a", "copy", "-movflags", "+faststart", OUT_MP4])
    except RuntimeError as e:
        raise RuntimeError(f"{e}\n\n烧字幕失败。不带字幕的版本在 {CLEAN_MP4}（左侧「文件」面板 output 里可以下载）；"
                           "改好 ①-2 的字幕设置后，运行 ①-2 再运行 ⑥ 只重做字幕") from None
    used = sorted({ln.split("->", 1)[1].strip() for ln in r.stderr.splitlines() if "fontselect:" in ln and "->" in ln})
    print(f"字幕已烧进画面：{n} 段，字体「{SUB_FONT}」（字体内部名称 {family}），效果「{SUB_EFFECT}」，"
          f"耗时 {time.time() - t0:.0f} 秒")
    if used:
        print("  ffmpeg 实际用的字体：" + "；".join(used))
    SUB_LAST_SIG = sub_signature()
    return [OUT_MP4, SUB_SRT]


ZIP_ALL = f"{OUT_DIR}/talking_avatar_outputs.zip"   # ⑤（去背关闭时）或 ⑤-2 打包下载的这次全部文件
ZIP_RESUB = f"{OUT_DIR}/talking_avatar_resub.zip"   # ⑥ 重做字幕后打包下载的新成品和字幕


def output_files(extra=()):
    """⑤／⑤-2 要下载的文件（存在的才算）：成品 mp4；开字幕时加 srt 和不带字幕的 mp4；再加 extra（⑤-2 的透明 WebM、绿幕图）。"""
    fs = [OUT_MP4] + ([SUB_SRT, CLEAN_MP4] if SUBTITLE else []) + list(extra)
    return [f for f in fs if os.path.exists(f)]


def download_once(paths, zip_path):
    """每格最多只触发一次下载：Chrome 会拦截同一个页面接连自动触发的多个下载（第一个之后的全被挡）。
    只有一个文件就直接下载；多个就先打包成 zip_path 再下载（视频本来就压缩过，zip 只打包不再压缩）。"""
    from google.colab import files
    paths = [p for p in dict.fromkeys(paths) if os.path.exists(p)]
    if not paths:
        print("（没有可以下载的文件）")
        return None
    target = paths[0]
    if len(paths) > 1:
        target = zip_path
        with zipfile.ZipFile(target, "w", zipfile.ZIP_STORED) as zf:
            for p in paths:
                zf.write(p, os.path.basename(p))
        print(f"已打包 {os.path.basename(target)}，解压后是：" + "、".join(os.path.basename(p) for p in paths))
    print(f"浏览器会自动下载 {os.path.basename(target)}；没弹出来：网址栏右边出现下载被拦截的图标时点它选允许，"
          "或从左侧「文件」面板的 output 文件夹下载")
    files.download(target)
    return target


if SUBTITLE:
    _tl, _dir, _family, _bold = prepare_subtitles()
    show_status(f"✅ 字幕准备完成：{len(_tl)} 句，字体「{SUB_FONT}」（字体内部名称 {_family}），效果「{SUB_EFFECT}」")
else:
    show_status("✅ 字幕：关闭（①-2 的 SUBTITLE = False），⑤ 会输出不带字幕的成品")

## ④ 对嘴（ComfyUI + InfiniteTalk）
- ComfyUI 固定 **v0.43.0**；模型 6 个文件全是 `.safetensors`，合计约 30GB，每次开机从 HuggingFace 下载（commit 已固定）。
- 画面尺寸由 ①-2 的 `ASPECT` 决定（默认 9:16 竖屏 480×832；作者实测过 9:16 和 1:1，16:9、4:5、3:4 未实测），6 步；角色图先按 `CROP_POS` 裁成同样比例（默认保留上方、裁掉下方）再缩放。
- ①-2 的 `BG_REMOVE` 选「绿幕」时，④-3 先用 BiRefNet 把角色图的背景换成纯绿色，跑完就释放显存，之后才启动 ComfyUI。
- 每段 81 帧（25fps，约 3.24 秒）；配音更长时一段接一段延伸（每段和前面重叠 9 帧当动作参考），最后裁回配音长度。
- 作者实测：21.5 秒的成品，这一步约 36 分钟。每 30 秒会打印一次进度。

In [ ]:
# @title ④-1 安装 ComfyUI { display-mode: "form" }
# @markdown 固定版本 v0.43.0。不用改。作者两次实测约 25～28 秒。
COMFY = "/content/ComfyUI"
COMFY_TAG = "v0.43.0"   # 节点输入名称按这个版本的源码写，换版本可能要改节点图
t0 = time.time()
if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1", "--branch", COMFY_TAG, "https://github.com/comfyanonymous/ComfyUI", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)
T_INSTALL = time.time() - t0
show_status(f"✅ ComfyUI {COMFY_TAG} 安装完成，{T_INSTALL:.0f} 秒")

In [ ]:
# @title ④-2 下载模型 { display-mode: "form" }
# @markdown 只下载 `.safetensors`，6 个文件约 30GB（来源与文件名照 ComfyUI 官方 InfiniteTalk 教程，commit 已固定）；已经下载过的会跳过。不用改。作者两次实测下载约 92 秒。
import shutil
from huggingface_hub import hf_hub_download

# (仓库, 文件路径, 固定的 commit, ComfyUI models 子目录)
MODELS = [
    ("Kijai/WanVideo_comfy_fp8_scaled", "I2V/Wan2_1-I2V-14B-480p_fp8_e4m3fn_scaled_KJ.safetensors",
     "033a4e487f60220b3d6e469599a6aebc46e13cee", "diffusion_models"),   # 16.6GB 底模 Wan2.1 I2V 14B 480P fp8
    ("Comfy-Org/Wan-2.1", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d", "text_encoders"),      # 6.7GB 文本编码器
    ("Comfy-Org/Wan-2.1", "split_files/model_patches/wan2.1_infiniteTalk_single_fp16.safetensors",
     "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d", "model_patches"),      # 5.1GB InfiniteTalk 单人
    ("Kijai/wav2vec2_safetensors", "wav2vec2-chinese-base_fp16.safetensors",
     "87847d3bc53702afda44078249e7c33e867827c4", "audio_encoders"),     # 0.19GB 中文 wav2vec2
    ("Kijai/WanVideo_comfy", "Wan2_1_VAE_bf16.safetensors",
     "8260d429d19fd7a72304cad059160b95d843913f", "vae"),                # 0.25GB
    ("Kijai/WanVideo_comfy", "Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
     "8260d429d19fd7a72304cad059160b95d843913f", "loras"),              # 0.74GB 加速 LoRA
]
print(f"磁盘剩余 {shutil.disk_usage('/content').free / 1e9:.0f}GB（需要约 30GB）")
show_status("⏳ 进行中，约 1.5 分钟：下载约 30GB 模型（作者两次实测约 92 秒；已下载过的会跳过）", busy=True)
t0 = time.time()
for repo, path, rev, folder in MODELS:
    name = os.path.basename(path)
    assert name.endswith(".safetensors"), name
    dst = f"{COMFY}/models/{folder}/{name}"
    if os.path.exists(dst):
        continue
    print("下载：", name)
    tmp = hf_hub_download(repo, path, revision=rev, local_dir="/content/hf_tmp")
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(tmp, dst)
T_DOWNLOAD = time.time() - t0
show_status(f"✅ 模型就绪，{T_DOWNLOAD:.0f} 秒")

In [ ]:
# @title ④-3 准备对嘴输入 { display-mode: "form" }
# @markdown 角色图按 ①-2 的画面比例和裁切位置裁切、缩放（去背选「绿幕」时再把背景换成纯绿色）；配音转成 16kHz 单声道给对嘴模型用。不用改。
import gc, math
import numpy as np
from PIL import Image

# WIDTH、HEIGHT 由 ①-2 的 ASPECT 决定（都是 16 的倍数）
STEPS = 6                  # 搭配 lightx2v 加速 LoRA，官方模板用 6 步
if not os.path.exists(VOICE_WAV):
    raise FileNotFoundError(f"找不到配音 {VOICE_WAV}：请先运行 ③")

# ---- 去背工具 BiRefNet（MIT）：只从 Hugging Face 官方仓库下载，commit 已固定。绿幕路线在本格用，逐帧路线在 ⑤-2 用 ----
BIREFNET_REPO = "ZhengPeng7/BiRefNet"
BIREFNET_REV = "e2bf8e4460fc8fa32bba5ea4d94b3233d367b0e4"
GREEN_RGB = (0, 255, 0)                        # 绿幕的颜色（纯绿 #00FF00）
INPUT_MASK = f"{OUT_DIR}/input_mask.png"      # 绿幕路线：角色图的人物遮罩（⑤-2 靠它在第一帧取绿幕的实际颜色）
PYMATTING_PIN = "pymatting==1.1.16"          # 绿幕路线：前景色估计（MIT 授权），去掉发丝里混进来的原背景颜色


def stop_comfyui():
    """关掉 ComfyUI，释放它占的显存（去背和对嘴模型不同时占显存）。"""
    subprocess.run(["pkill", "-f", "main.py --listen 127.0.0.1 --port 8188"])
    time.sleep(3)


def load_birefnet():
    """载入 BiRefNet（半精度、放到 GPU、推理模式）。用法照官方模型卡：transformers 的 AutoModelForImageSegmentation。"""
    # 只补装 BiRefNet 程序用到的套件；不装它的 requirements.txt（里面固定的 torch 版本会盖掉 Colab 的）
    run([sys.executable, "-m", "pip", "install", "-q", "kornia", "timm", "einops"])
    import torch
    from transformers import AutoModelForImageSegmentation
    t0 = time.time()
    m = AutoModelForImageSegmentation.from_pretrained(BIREFNET_REPO, revision=BIREFNET_REV, trust_remote_code=True)
    torch.set_float32_matmul_precision("high")
    m.to("cuda").train(False).half()   # train(False)＝推理模式
    print(f"BiRefNet 就绪（{BIREFNET_REPO} @ {BIREFNET_REV[:8]}），{time.time() - t0:.0f} 秒；目前显存占用：{gpu_used()}")
    return m


def birefnet_masks(m, images):
    """一批 PIL RGB 图 → 同尺寸的人物遮罩（L 模式：255＝人物、0＝背景）。照官方模型卡：缩成 1024×1024 推理，再缩回原尺寸。"""
    import torch
    from torchvision import transforms
    tf = transforms.Compose([transforms.Resize((1024, 1024)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    x = torch.stack([tf(im) for im in images]).to("cuda").half()
    with torch.no_grad():
        preds = m(x)[-1].sigmoid().float().cpu()
    return [transforms.ToPILImage()(p.squeeze()).resize(im.size, Image.BILINEAR) for p, im in zip(preds, images)]


def green_composite(img, mask):
    """把人物合成到纯绿底。先用 pymatting 的前景色估计把发丝等半透明处混进来的原背景颜色去掉
    （不然逆光发丝在绿底上会带着原背景的褐色／金色，扣绿后变成一圈毛边）；pymatting 装不上时退回直接合成。"""
    try:
        try:
            from pymatting import estimate_foreground_ml
        except ImportError:
            run([sys.executable, "-m", "pip", "install", "-q", PYMATTING_PIN])
            from pymatting import estimate_foreground_ml
        a = np.asarray(mask, np.float64)[..., None] / 255.0
        fg = estimate_foreground_ml(np.asarray(img, np.float64) / 255.0, a[..., 0])
        out = fg * a + np.array(GREEN_RGB, np.float64) / 255.0 * (1 - a)
        return Image.fromarray(np.clip(out * 255 + 0.5, 0, 255).astype(np.uint8))
    except Exception as e:
        print(f"（前景色估计失败，改成直接合成到绿底，发丝边缘可能带原背景的颜色：{e!r}）")
        return Image.composite(img, Image.new("RGB", img.size, GREEN_RGB), mask)


def free_gpu():
    import torch
    gc.collect()
    torch.cuda.empty_cache()


# ---- 裁切：只会裁一个方向（太宽裁左右、太高裁上下）；锚点 0＝保留左／上，0.5＝保留中间，1＝保留右 ----
CROP_ANCHOR = {"偏上（保留头部）": (0.5, 0.0), "置中": (0.5, 0.5), "偏左": (0.0, 0.0), "偏右": (1.0, 0.0)}
ax, ay = CROP_ANCHOR[CROP_POS]
IMG_NAME, AUD_NAME = "ta_host.png", "ta_voice.wav"
im = Image.open(IMAGE_FILE).convert("RGB")
w0, h0 = im.size
target = WIDTH / HEIGHT
if w0 / h0 > target:   # 太宽：裁左右
    cw = round(h0 * target)
    x0 = int((w0 - cw) * ax)
    box = (x0, 0, x0 + cw, h0)
else:                  # 太高：裁上下
    ch = round(w0 / target)
    y0 = int((h0 - ch) * ay)
    box = (0, y0, w0, y0 + ch)
im = im.crop(box)
if BG_MODE == "green":
    # 绿幕：先去背、把背景换成纯绿，再交给对嘴；BiRefNet 用完马上释放，ComfyUI 在 ④-4 才启动
    stop_comfyui()   # 重跑时上一次的 ComfyUI 可能还占着显存（④-4 本来就会重新启动它）
    bref = load_birefnet()
    mask = birefnet_masks(bref, [im])[0]
    del bref
    free_gpu()
    print("已释放 BiRefNet 显存，目前显存占用：", gpu_used())
    # 先缩到对嘴尺寸再估计前景色（大图直接算很慢，结果也要缩小）
    im = im.resize((WIDTH, HEIGHT), Image.LANCZOS)
    mask = mask.resize((WIDTH, HEIGHT), Image.LANCZOS)
    mask.save(INPUT_MASK)
    im = green_composite(im, mask)
elif os.path.exists(INPUT_MASK):
    os.remove(INPUT_MASK)   # 这次不是绿幕：删掉上一次的遮罩，免得 ⑤-2 误以为对嘴用的是绿幕图
im = im.resize((WIDTH, HEIGHT), Image.LANCZOS)
im.save(f"{COMFY}/input/{IMG_NAME}")
print(f"图：{w0}x{h0} → 裁切 {box}（{CROP_POS}）→ {WIDTH}x{HEIGHT}")
if BG_MODE == "green":
    im.save(f"{OUT_DIR}/input_greenscreen.png")
    print(f"绿幕图（对嘴用的输入）：{OUT_DIR}/input_greenscreen.png；人物边缘被吃掉、或背景没换干净，"
          "可以在 ①-2 改用「逐帧」，运行 ①-2 再从 ④-3 重跑")
    from IPython.display import display
    display(im.resize((max(1, WIDTH * 240 // HEIGHT), 240)))

_lip = []
if LAYOUT_STREAM:
    # 上主播下录影：对嘴只生到最后一句结束后 ANCHOR_TAIL_SEC 秒（这段是静音，省计算单元）；成片比这长的部分，⑤ 只拿这段静音画面来回循环
    _vend = probe_media(VOICE_WAV)["dur"]
    _lip = ["-af", "apad", "-t", f"{min(_vend + ANCHOR_TAIL_SEC, max(REC_DUR, _vend)):.3f}"]
run(["ffmpeg", "-v", "error", "-y", "-i", VOICE_WAV, *_lip, "-ac", "1", "-ar", "16000", f"{COMFY}/input/{AUD_NAME}"])
AUDIO_SEC = float(run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                       f"{COMFY}/input/{AUD_NAME}"]).stdout)

# 每段 81 帧；超过就接延伸段（重叠 9 帧当动作参考），最后裁回配音长度
FPS, CHUNK, MOTION = 25, 81, 9
NEED_FRAMES = math.ceil(AUDIO_SEC * FPS)
N_CHUNKS = 1 + max(0, math.ceil((NEED_FRAMES - CHUNK) / (CHUNK - MOTION)))
print(f"配音 {AUDIO_SEC:.2f} 秒 → {NEED_FRAMES} 帧，分 {N_CHUNKS} 段生成")
if LAYOUT_STREAM:
    print(f"上主播下录影：最后一句在 {_vend:.2f} 秒结束，对嘴生到 {AUDIO_SEC:.2f} 秒；成片 {max(REC_DUR, _vend):.2f} 秒"
          + (f"，之后的主播画面用最后一句结束后的 {ANCHOR_TAIL_SEC:g} 秒静音画面来回循环" if max(REC_DUR, _vend) > AUDIO_SEC + 0.02 else ""))
if AUDIO_SEC > 22:
    print("⚠ 提示：作者实测过的最长成品是 21.5 秒（8 段，约 36 分钟）；更长时耗时按段数增加，系统内存够不够未验证")
show_status(f"✅ 对嘴输入准备好了：画面 {WIDTH}x{HEIGHT}，配音 {AUDIO_SEC:.2f} 秒，分 {N_CHUNKS} 段")

In [ ]:
# @title ④-4 启动 ComfyUI { display-mode: "form" }
# @markdown 在后台启动 ComfyUI（只监听 127.0.0.1，不开网页界面）。不用改。作者两次实测启动 18～27 秒。
import json, threading, urllib.request, urllib.error

API = "http://127.0.0.1:8188"
args = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"]
if LOW_VRAM:
    args.append("--novram")


def _serve():
    # 在后台线程里跑 ComfyUI，输出写进 /content/comfyui.log
    with open("/content/comfyui.log", "w") as log:
        subprocess.run(args, cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)


def _post(path, payload):
    req = urllib.request.Request(API + path, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        return json.load(urllib.request.urlopen(req))
    except urllib.error.HTTPError as e:
        raise RuntimeError(e.read().decode()) from None


def _get(path):
    return json.load(urllib.request.urlopen(API + path))


# 重跑时先关掉上一次留下的 ComfyUI（重启会话后它可能还占着显存和端口）
subprocess.run(["pkill", "-f", "main.py --listen 127.0.0.1 --port 8188"])
time.sleep(3)
t0 = time.time()
server = threading.Thread(target=_serve, daemon=True)
server.start()
for _ in range(300):
    if not server.is_alive():
        raise RuntimeError("ComfyUI 启动失败，请看 /content/comfyui.log")
    try:
        _get("/system_stats")
        break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(1)
else:
    raise RuntimeError("等了 300 秒还没启动，请看 /content/comfyui.log")
T_BOOT = time.time() - t0
if "WanInfiniteTalkToVideo" not in _get("/object_info/WanInfiniteTalkToVideo"):
    raise RuntimeError("这个版本的 ComfyUI 没有 WanInfiniteTalkToVideo 节点")
show_status(f"✅ ComfyUI 已启动，{T_BOOT:.0f} 秒；LOW_VRAM={LOW_VRAM}")

In [ ]:
# @title ④-5 生成对嘴视频 { display-mode: "form" }
# @markdown **最久的一步**，每 30 秒打印一次进度；进度条走完 6 步是一段，之后回到 0/6 做下一段，正常。不用改。
# @markdown 作者实测：示例文稿（配音 10.88 秒，4 段）约 16 分钟；21.5 秒（8 段）约 36 分钟；文稿越长越久。
M = {folder: os.path.basename(path) for _, path, _, folder in MODELS}


def build_graph():
    # 节点与参数照官方模板 video_wan2_1_infinitetalk.json（单人）：lightx2v LoRA、euler、normal、cfg 1、audio_scale 1
    g = {}

    def add(cls, **inputs):
        nid = str(len(g) + 1)
        g[nid] = {"class_type": cls, "inputs": inputs}
        return nid

    unet = add("UNETLoader", unet_name=M["diffusion_models"], weight_dtype="default")
    model = add("LoraLoaderModelOnly", model=[unet, 0], lora_name=M["loras"], strength_model=1.0)
    clip = add("CLIPLoader", clip_name=M["text_encoders"], type="wan", device="default")
    vae = add("VAELoader", vae_name=M["vae"])
    patch = add("ModelPatchLoader", name=M["model_patches"])
    aenc = add("AudioEncoderLoader", audio_encoder_name=M["audio_encoders"])
    img = add("LoadImage", image=IMG_NAME)
    aud = add("LoadAudio", audio=AUD_NAME)
    aemb = add("AudioEncoderEncode", audio_encoder=[aenc, 0], audio=[aud, 0])
    pos = add("CLIPTextEncode", clip=[clip, 0], text=PROMPT)
    neg = add("ConditioningZeroOut", conditioning=[pos, 0])
    sampler = add("KSamplerSelect", sampler_name="euler")
    frames = None
    for i in range(N_CHUNKS):
        it_in = dict(mode="single_speaker", model=[model, 0], model_patch=[patch, 0], positive=[pos, 0],
                     negative=[neg, 0], vae=[vae, 0], width=WIDTH, height=HEIGHT, length=CHUNK,
                     start_image=[img, 0], audio_encoder_output_1=[aemb, 0], motion_frame_count=MOTION,
                     audio_scale=1.0)
        if frames is not None:
            it_in["previous_frames"] = [frames, 0]   # 延伸段：接在前面所有帧之后
        it = add("WanInfiniteTalkToVideo", **it_in)
        guider = add("CFGGuider", model=[it, 0], positive=[it, 1], negative=[it, 2], cfg=1.0)
        sigmas = add("BasicScheduler", model=[it, 0], scheduler="normal", steps=STEPS, denoise=1.0)
        noise = add("RandomNoise", noise_seed=SEED + i)
        lat = add("SamplerCustomAdvanced", noise=[noise, 0], guider=[guider, 0], sampler=[sampler, 0],
                  sigmas=[sigmas, 0], latent_image=[it, 3])
        dec = add("VAEDecode", samples=[lat, 0], vae=[vae, 0])
        if frames is None:
            frames = dec
        else:
            new = add("ImageFromBatch", image=[dec, 0], batch_index=MOTION, length=4096)   # 去掉重叠的动作参考帧
            frames = add("ImageBatch", image1=[frames, 0], image2=[new, 0])
    final = add("ImageFromBatch", image=[frames, 0], batch_index=0, length=NEED_FRAMES)
    video = add("CreateVideo", images=[final, 0], fps=float(FPS), audio=[aud, 0])
    save = add("SaveVideo", video=[video, 0], filename_prefix="talking_avatar/raw",
               format="auto", **{"format.codec": "auto"})
    return g, save


show_status(f"⏳ 进行中，这次分 {N_CHUNKS} 段（作者实测：4 段约 16 分钟、8 段约 36 分钟，段数越多越久）；"
            "每 30 秒打印一次进度", busy=True)
graph, save_id = build_graph()
t0 = time.time()
pid = _post("/prompt", {"prompt": graph})["prompt_id"]
last = 0
while True:
    hist = _get(f"/history/{pid}")
    if pid in hist:
        break
    if not server.is_alive():
        raise RuntimeError("ComfyUI 进程结束了（多半是系统内存或显存不够），请看 /content/comfyui.log")
    if time.time() - last >= 30:
        last = time.time()
        with open("/content/comfyui.log", "rb") as f:
            f.seek(max(0, os.path.getsize("/content/comfyui.log") - 400))
            tail = f.read().decode(errors="ignore").replace("\r", "\n").strip().splitlines()
        print(f"[{time.time() - t0:5.0f} 秒] {tail[-1][:150] if tail else ''}")
    time.sleep(2)
T_GEN = time.time() - t0

status = hist[pid].get("status", {})
if status.get("status_str") == "error":
    raise RuntimeError(json.dumps(status.get("messages"), ensure_ascii=False)[:3000])
v = hist[pid]["outputs"][save_id]["images"][0]
RAW_MP4 = os.path.join(COMFY, "output", v["subfolder"], v["filename"])
show_status(f"✅ 对嘴完成：{RAW_MP4}，生成耗时 {T_GEN / 60:.1f} 分钟")

In [ ]:
# @title ⑤ 合成输出 { display-mode: "form" }
# @markdown 画面原样保留，音轨换回原始配音（ComfyUI 输出里的是转过的 16kHz 版）；字幕开启时再把字幕烧进画面。①-2 的 `LAYOUT` 选「上主播下录影」时：先把主播片和录影合成 1080×1920（②-2 的 `compose_stream`），再烧字幕。不用改。
# @markdown **跑完后只想调版面**（①-2 的 `REC_CROP_TOP`、`REC_CROP_BOTTOM`、`REC_CENTER_X`、`HOST_CROP_TOP`、`BG_BLUR`、`BG_DIM`、`REC_AUDIO`）：改 ①-2 → 运行 ①-2 → 再运行本格就好，不用重跑配音和对嘴（会再下载一次 zip）；换了 BGM（①-2 的 `BGM_PATH` 或 ①-3 上传）时，先运行 ②-2 再运行本格。（这个做法只在作者电脑上模拟验证过）
from IPython.display import Video, display

# 先合成不带字幕的版本 CLEAN_MP4（⑥ 换字幕样式时拿它重烧），再按 ①-2 的字幕设置做出成品 OUT_MP4（路径定义在 ③-2）
_extra = []
if LAYOUT_STREAM:
    # 只重跑本格时：录影、BGM 的路径是 ②-2 检查过、存在 REC_FILE／BGM_FILE 里的；①-2／①-3 改了路径却没重跑 ②-2 就先停下来
    if globals().get("REC_FILE") != resolve(REC_PATH):
        raise RuntimeError("①-2／①-3 的 REC_PATH（或 ①-2 的 LAYOUT）和上次运行 ②-2 时不同。换了录影的话，台词秒数和对嘴长度是照旧录影排的，"
                           "请重新全部运行；只是路径写法不同，就先运行 ②-2 再运行本格")
    if BGM_FILE != (resolve(BGM_PATH) if BGM_PATH.strip() else None):
        raise RuntimeError("①-2／①-3 的 BGM_PATH 和上次运行 ②-2 时不同：请先运行 ②-2（会检查新的 BGM），再运行本格")
    HOST_MP4 = f"{OUT_DIR}/talking_avatar_host.mp4"   # 只有主播的对嘴片（不带字幕），一起打包给你
    run(["ffmpeg", "-v", "error", "-y", "-i", RAW_MP4, "-i", VOICE_WAV, "-map", "0:v", "-map", "1:a",
         "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", HOST_MP4])
    t_lay = time.time()
    LAYOUT_INFO = compose_stream(HOST_MP4, VOICE_WAV, REC_FILE, CLEAN_MP4, load_timeline(),
                                 rec_crop_top=REC_CROP_TOP, rec_crop_bottom=REC_CROP_BOTTOM, rec_center_x=REC_CENTER_X,
                                 host_crop_top=HOST_CROP_TOP, rec_audio=REC_AUDIO, bgm=BGM_FILE,
                                 bg_blur=BG_BLUR, bg_dim=BG_DIM, work_dir=OUT_DIR)
    print(f"上主播下录影合成完成：{STREAM_W}x{STREAM_H}，{LAYOUT_INFO['dur']:.2f} 秒，耗时 {time.time() - t_lay:.0f} 秒")
    _hs, _hc, _rc = LAYOUT_INFO["host_scaled"], LAYOUT_INFO["host_crop"], LAYOUT_INFO["rec_crop"]
    print(f"  上格主播：对嘴片放大到 {_hs[0]}x{_hs[1]}，取 y={_hc[1]}–{_hc[1] + _hc[3]}（HOST_CROP_TOP={HOST_CROP_TOP}%）")
    print(f"  下格录影：取录影 x={_rc[0]}–{_rc[0] + _rc[2]}、y={_rc[1]}–{_rc[1] + _rc[3]}"
          + ("，录影比下格窄，两侧模糊补边" if LAYOUT_INFO["rec_side_fill"] else ""))
    if LAYOUT_INFO["pingpong_sec"] > 0:
        print(f"  对嘴片比成片短 {LAYOUT_INFO['pingpong_sec']:.2f} 秒：这段主播画面用最后一句结束后的静音画面（最多 {PINGPONG_SEC:g} 秒）来回循环")
    if LAYOUT_INFO["freeze_sec"] > 0:
        print(f"  配音比录影长 {LAYOUT_INFO['freeze_sec']:.2f} 秒：这段录影停在最后一格")
    print(f"  声音：主播配音" + "".join({"rec": "＋录影原音", "bgm": "＋BGM"}[b] for b in LAYOUT_INFO["beds"])
          + ("（主播讲话时自动压低）" if LAYOUT_INFO["beds"] else ""))
    _extra = [HOST_MP4]
else:
    run(["ffmpeg", "-v", "error", "-y", "-i", RAW_MP4, "-i", VOICE_WAV, "-map", "0:v", "-map", "1:a",
         "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", CLEAN_MP4])
make_final_video()

print("=" * 60)
print(f"GPU：{GPU_NAME}")
if LAYOUT_STREAM:
    print(f"成品：{OUT_MP4}（上主播下录影 {STREAM_W}x{STREAM_H}，{LAYOUT_INFO['dur']:.2f} 秒；对嘴 {AUDIO_SEC:.2f} 秒，{N_CHUNKS} 段）")
    print(f"只有主播的对嘴片：{HOST_MP4}（{WIDTH}x{HEIGHT}，不带字幕）")
else:
    print(f"成品：{OUT_MP4}（{WIDTH}x{HEIGHT}，配音 {AUDIO_SEC:.2f} 秒，{N_CHUNKS} 段）")
if SUBTITLE:
    print(f"字幕：已烧进画面，另附 {SUB_SRT} 和不带字幕的 {CLEAN_MP4}；想换字幕样式用 ⑥（不用重跑配音和对嘴）")
if BG_MODE == "green":
    print("去背（绿幕）：上面这份 mp4 的背景是绿色的；透明背景的 talking_avatar_alpha.webm 由下一格 ⑤-2 产生")
elif BG_MODE == "frames":
    print("去背（逐帧）：透明背景的 talking_avatar_alpha.webm 由下一格 ⑤-2 逐帧处理产生")
print(f"耗时：配音 {T_VOICE:.0f} 秒、装 ComfyUI {T_INSTALL:.0f} 秒、下载模型 {T_DOWNLOAD:.0f} 秒、"
      f"启动 {T_BOOT:.0f} 秒、对嘴 {T_GEN:.0f} 秒；从 ② 到现在共 {(time.time() - T_START) / 60:.1f} 分钟")
if BG_MODE != "off":
    print("下载：这次的文件等 ⑤-2 做完透明背景版后，一起打包成 talking_avatar_outputs.zip 下载"
          "（一次流程只自动下载一次，免得浏览器拦截）")
print("=" * 60)
if BG_MODE == "off":
    download_once(output_files(_extra), ZIP_ALL)   # 去背关闭时 ⑤-2 会跳过，这次的文件在这里一次下载
if LAYOUT_STREAM:   # 1080×1920 成片较大，嵌进 notebook 会让页面变慢：只提供下载
    print("上主播下录影：成片不在这里预览，请看下载的 talking_avatar_outputs.zip 里的 talking_avatar.mp4")
else:
    display(Video(OUT_MP4, embed=True, width=360))
show_status(f"✅ 合成完成：{OUT_MP4}")

## ⑤-2 去背输出（①-2 的 `BG_REMOVE` 选「绿幕」或「逐帧」才做；选「关」时自动跳过）
- 输出 `/content/output/talking_avatar_alpha.webm`：VP9 视频＋透明通道（`yuva420p`）＋原始配音（Opus），**不烧字幕**，给之后放进别的模板合成用（需要字幕可以用 `talking_avatar.srt`）。
- 「绿幕」：ffmpeg `chromakey` 扣掉绿色，再用 `despill` 压掉边缘反射的绿光。绿色的实际色值从第一帧的背景取（对嘴模型可能让绿色有点偏）。**这时 `talking_avatar.mp4` 的背景是绿色的。** ④-3 做的绿幕图 `input_greenscreen.png` 会一并打包下载，边缘不满意时拿它对照：绿幕图上就有毛边＝去背那一步的问题，绿幕图干净＝扣绿这一步的问题。
- 「逐帧」：先关掉 ComfyUI 释放显存，再对成品的每一帧跑 BiRefNet 得到透明遮罩，最后和画面合在一起。比绿幕慢，前后帧的边缘可能轻微闪动。
- 绿幕效果不满意（绿边、背景没扣干净）：最干净的做法是 ①-2 改成「逐帧」后重新全部运行；不想重跑对嘴，也可以 ①-2 改成「逐帧」→ 运行 ①-2 → 只运行本格（拿绿底的视频逐帧去背，发丝边缘可能还带一点绿）。
- 透明效果要用支持透明 WebM 的软件才看得到；有些播放器会显示成黑底。
- 下载：开了去背时 ⑤ 不下载，等本格做完，把这次的所有文件（⑤ 的成品、字幕，本格的透明视频、绿幕图）打包成一个 `talking_avatar_outputs.zip` 一次下载（浏览器会拦截同一页接连的多个下载）。
- 这一格出错不影响 ⑤ 已经做好的 mp4 和字幕，出错时也会照样把它们打包下载。

In [ ]:
# @title ⑤-2 去背输出 { display-mode: "form" }
# @markdown 透明背景的 `talking_avatar_alpha.webm`（VP9＋透明通道，不烧字幕）；①-2 的 `BG_REMOVE` 选「关」时自动跳过。不用改。作者实测：绿幕约 40 秒；逐帧 354 帧整格约 141 秒。
import os, shutil, subprocess, time
import numpy as np
from PIL import Image

OUT_WEBM = f"{OUT_DIR}/talking_avatar_alpha.webm"
FRAME_DIR = f"{OUT_DIR}/matte_frames"   # 逐帧路线：成品拆出来的每一帧（做完就删）
MATTE_DIR = f"{OUT_DIR}/matte_masks"    # 逐帧路线：每一帧的透明遮罩（灰度 PNG，做完就删）
KEY_SIMILARITY = 0.08   # 绿幕：颜色和绿幕多接近就完全扣掉（0～1）；越大扣得越多，太大会把发丝和人物身上偏绿的地方也扣掉
KEY_BLEND = 0.22        # 绿幕：半透明过渡的范围（0～1）；越大发丝边缘越柔，太小发丝外围会变成一圈深色硬边
                        #   白色、灰色（白发、白衣）和绿色的距离只有 0.3 多，KEY_SIMILARITY＋KEY_BLEND 超过 0.3 就可能让白衣变半透明
DESPILL_MIX = 0.5       # 绿幕：去绿光时绿色压到红、蓝的哪个比例（ffmpeg despill 的 mix；0.5＝压到红蓝平均）
DESPILL_BLUE = -0.1     # 绿幕：去绿光时顺便压一点蓝（ffmpeg despill 的 blue）；金属反光的绿光去掉后不会变成蓝青色
GREEN_PREVIEW = f"{OUT_DIR}/input_greenscreen.png"   # ④-3 存的绿幕图（对嘴用的输入），本格一并打包下载
MATTE_BATCH = 4         # 逐帧：每次送几帧给 BiRefNet
# VP9＋透明通道：yuva420p；-b:v 0 加 -crf 30＝固定画质模式；libvpx 编透明视频要关掉 alt-ref（-auto-alt-ref 0）
WEBM_ARGS = ["-c:v", "libvpx-vp9", "-pix_fmt", "yuva420p", "-b:v", "0", "-crf", "30", "-row-mt", "1",
             "-auto-alt-ref", "0", "-c:a", "libopus", "-b:a", "128k"]


def probe_size(path):
    wh = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height",
              "-of", "csv=p=0:s=x", path]).stdout.strip()
    w, h = (int(v) for v in wh.split("x"))
    return w, h


def sample_key_color(w, h):
    """绿幕的实际颜色：第一帧里、角色图遮罩确定是背景的地方取颜色中位数；取不到或不像绿色就用纯绿 #00FF00。"""
    try:
        r = subprocess.run(["ffmpeg", "-v", "error", "-i", CLEAN_MP4, "-frames:v", "1",
                            "-f", "rawvideo", "-pix_fmt", "rgb24", "-"], capture_output=True, check=True)
        frame = np.frombuffer(r.stdout, np.uint8)[: w * h * 3].reshape(h, w, 3)
        bg = np.asarray(Image.open(INPUT_MASK).convert("L").resize((w, h))) < 8
        if bg.sum() < 500:
            raise ValueError("确定是背景的像素太少")
        rr, gg, bb = (int(v) for v in np.median(frame[bg], axis=0).round())
        if gg < max(rr, bb) + 40:
            raise ValueError(f"取到的颜色 #{rr:02X}{gg:02X}{bb:02X} 不像绿色")
    except Exception as e:
        print(f"（取不到绿幕的实际颜色，改用纯绿 #00FF00：{e}）")
        return "0x00FF00"
    print(f"绿幕实际颜色（第一帧背景的中位数）：#{rr:02X}{gg:02X}{bb:02X}")
    return f"0x{rr:02X}{gg:02X}{bb:02X}"


def make_mattes():
    """逐帧路线：成品拆成一张张 PNG → 每张跑 BiRefNet → 遮罩存成同名灰度 PNG（MATTE_DIR）；回传帧数。"""
    stop_comfyui()   # 对嘴已经做完，关掉 ComfyUI 把显存让给 BiRefNet（之后要重跑对嘴得从 ④-4 开始）
    print("已关闭 ComfyUI，目前显存占用：", gpu_used())
    for d in (FRAME_DIR, MATTE_DIR):
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d)
    # -vsync 0：一帧对一帧原样拆出来，不补帧也不丢帧
    run(["ffmpeg", "-v", "error", "-i", CLEAN_MP4, "-vsync", "0", f"{FRAME_DIR}/%06d.png"])
    names = sorted(os.listdir(FRAME_DIR))
    if not names:
        raise RuntimeError(f"没有从 {CLEAN_MP4} 拆出任何一帧")
    bref = load_birefnet()
    t0 = last = time.time()
    try:
        for k in range(0, len(names), MATTE_BATCH):
            batch = names[k:k + MATTE_BATCH]
            imgs = [Image.open(f"{FRAME_DIR}/{nm}").convert("RGB") for nm in batch]
            for nm, mk in zip(batch, birefnet_masks(bref, imgs)):
                mk.save(f"{MATTE_DIR}/{nm}")
            if time.time() - last >= 30:
                last = time.time()
                print(f"[{time.time() - t0:5.0f} 秒] 已去背 {k + len(batch)}/{len(names)} 帧")
    finally:
        del bref
        free_gpu()
    print(f"逐帧去背完成：{len(names)} 帧，耗时 {time.time() - t0:.0f} 秒；已释放 BiRefNet 显存，目前显存占用：{gpu_used()}")
    return len(names)


if BG_MODE == "off":
    show_status("✅ 去背：关闭（①-2 的 BG_REMOVE = 关），跳过；这次的文件已经在 ⑤ 下载")
else:
    if any(nm not in globals() for nm in ("load_birefnet", "birefnet_masks", "stop_comfyui", "free_gpu")):
        raise RuntimeError("缺少 ④-3 定义的去背函数：请先运行 ④-3 再运行本格")
    if not os.path.exists(CLEAN_MP4):
        raise FileNotFoundError(f"找不到 {CLEAN_MP4}：要先跑完 ⑤")
    _done = []   # 本格做成的文件；不管本格成功还是出错，最后都连同 ⑤ 的成品一起打包，只下载一次
    try:
        _enc = run(["ffmpeg", "-hide_banner", "-encoders"]).stdout
        for _e in ("libvpx-vp9", "libopus"):
            if _e not in _enc:
                raise RuntimeError(f"这台机器的 ffmpeg 没有 {_e} 编码器，做不了透明 WebM；⑤ 的 mp4 不受影响")
        W_, H_ = probe_size(CLEAN_MP4)
        t_bg = time.time()
        if BG_MODE == "green":
            if not os.path.exists(INPUT_MASK):
                raise FileNotFoundError(f"找不到 {INPUT_MASK}：这次对嘴不是用绿幕图做的。"
                                        "可以在 ①-2 改用「逐帧」，运行 ①-2 再运行本格（不用重跑对嘴）；或从 ④-3 重跑走绿幕")
            key = sample_key_color(W_, H_)
            # 先转成 yuva444p 再扣色（色度不降采样，边缘比较细）；despill 压掉边缘反射的绿光
            vf = (f"format=yuva444p,chromakey={key}:{KEY_SIMILARITY}:{KEY_BLEND},"
                  f"format=rgba,despill=type=green:mix={DESPILL_MIX}:expand=0:blue={DESPILL_BLUE},format=yuva420p")
            run(["ffmpeg", "-hide_banner", "-v", "error", "-y", "-i", CLEAN_MP4, "-vf", vf,
                 "-map", "0:v", "-map", "0:a", *WEBM_ARGS, OUT_WEBM])
        else:
            make_mattes()
            run(["ffmpeg", "-hide_banner", "-v", "error", "-y", "-i", CLEAN_MP4,
                 "-framerate", str(FPS), "-i", f"{MATTE_DIR}/%06d.png",
                 "-filter_complex", "[1:v]format=gray[a];[0:v][a]alphamerge,format=yuva420p[v]",
                 "-map", "[v]", "-map", "0:a", *WEBM_ARGS, OUT_WEBM])
            for d in (FRAME_DIR, MATTE_DIR):
                shutil.rmtree(d, ignore_errors=True)
        _done.append(OUT_WEBM)
        print("=" * 60)
        print(f"透明背景版：{OUT_WEBM}（{W_}x{H_}，VP9＋透明通道，不带字幕），去背「{BG_REMOVE}」，"
              f"本格耗时 {time.time() - t_bg:.0f} 秒")
        if BG_MODE == "green":
            print("头发边缘有绿边、或背景没扣干净：在 ①-2 改用「逐帧」后重新全部运行最干净；"
                  "不想重跑对嘴，也可以改「逐帧」→ 运行 ①-2 → 只运行本格（发丝边缘可能还带一点绿）")
        print("=" * 60)
    finally:
        if BG_MODE == "green" and os.path.exists(GREEN_PREVIEW):
            _done.append(GREEN_PREVIEW)   # 绿幕图：边缘没扣干净时，拿它对照是去背那一步还是扣绿那一步的问题
        download_once(output_files(_done), ZIP_ALL)
    show_status(f"✅ 去背完成：{OUT_WEBM}")

## ⑥ 只重做字幕（换字幕样式时用）
跑完一次后，想换字幕的字体、字号、效果、位置、颜色，或打开／关闭字幕：
1. 改 ①-2 设置格里「字幕」那一段（自定义字体也可以用 ①-3 上传）。
2. 运行 ①-2 设置格（点格子左边的 ▶；用 ①-3 上传了新字体的话，再运行 ①-3）。
3. 运行下面这一格，会自动下载一个 `talking_avatar_resub.zip`（里面是新的 `talking_avatar.mp4` 和 `talking_avatar.srt`；字幕关闭时直接下载 `talking_avatar.mp4`）。

不会重跑配音和对嘴：拿 `/content/output/` 里不带字幕的版本 `talking_avatar_nosub.mp4` 和字幕时间轴 `subtitle_timeline.json` 重新烧字幕。
- 「全部运行」时这一格会自动跳过（字幕设置没变）。
- 中途「重启会话」过：先依次运行 ①、①-2、②、②-2、③-2（②-2、③-2 很快；①-3 上传过的文件会自动沿用），再运行这一格。**不要**运行 ③-1，会重新配音。
- **删除运行时后文件就没了，不能再重做字幕**，所以换样式要在删除运行时之前做。
- 字幕文字固定用文稿原文；想改字幕文字，要改 ① 文稿后重新全部运行。
- 透明背景的 `talking_avatar_alpha.webm` 本来就不带字幕，⑥ 不会动它。

In [ ]:
# @title ⑥ 只重做字幕 { display-mode: "form" }
# @markdown 用 ①-2 现在的字幕设置，拿不带字幕的版本重烧（不重跑配音和对嘴）。「全部运行」时字幕设置没变会自动跳过。作者实测每次约 2～5 秒。
import os
from IPython.display import Video, display

if any(n not in globals() for n in ("show_status", "run", "resolve", "make_final_video", "download_once")):
    raise RuntimeError("缺少前面格子定义的函数（多半是重启过会话）：请先依次运行 ①、①-2、②、②-2、③-2，再运行本格。"
                       "不要运行 ③-1（会重新配音）")
if not os.path.exists(CLEAN_MP4):
    raise FileNotFoundError(f"找不到不带字幕的版本 {CLEAN_MP4}：要先完整跑完一次（到 ⑤）。"
                            "删除过运行时的话文件已经没了，只能重新全部运行")
if globals().get("SUB_LAST_SIG") == sub_signature():
    show_status("✅ 字幕设置和现在的成品一样，跳过（「全部运行」时这一格本来就会跳过）。\n"
                "要换样式：改 ①-2 的字幕设置 → 运行 ①-2 → 再运行本格")
else:
    download_once(make_final_video(), ZIP_RESUB)   # 只触发一次下载：开字幕时 mp4＋srt 打包成 zip，关字幕时直接下载 mp4
    if globals().get("LAYOUT_STREAM"):   # 1080×1920 成片较大，嵌进 notebook 会让页面变慢：只提供下载
        print("上主播下录影：成片不在这里预览，请看刚下载的 talking_avatar_resub.zip 里的 talking_avatar.mp4"
              "（字幕关闭时直接下载的就是 talking_avatar.mp4）")
    else:
        display(Video(OUT_MP4, embed=True, width=360))
    show_status("✅ 字幕重做完成")

## 跑完之后（重要）
- 确认浏览器下载文件夹里有 `talking_avatar_outputs.zip`（一次流程只自动下载这一个文件），解压后是 `talking_avatar.mp4`；开字幕时还有 `talking_avatar.srt` 和不带字幕的 `talking_avatar_nosub.mp4`，开去背时还有 `talking_avatar_alpha.webm`，选「绿幕」时还有绿幕图 `input_greenscreen.png`。字幕和去背都关时只有一个文件，直接下载 `talking_avatar.mp4`。版面选「上主播下录影」时还有只有主播的 `talking_avatar_host.mp4`。
- 没下载：Chrome 网址栏右边出现下载被拦截的图标时，点它选允许；或从左侧「文件」面板 `output/` 里手动下载。
- 想换字幕样式：要在删除运行时**之前**用 ⑥ 只重做字幕。
- **「代码执行程序 → 断开连接并删除运行时」**（英文界面：Runtime → Disconnect and delete runtime）。**闲置也会扣计算单元。** 第一次点可能先弹出警告，确认后要再点一次删除。
- 爆显存（报错里有 `OutOfMemory` / `OOM`）：「代码执行程序 → 重启会话」→ ①-2 勾选 `LOW_VRAM` → 全部运行（已装的 ComfyUI 和已下载的模型会跳过）。